In [8]:
# ============================================================
# KKT anomaly detection pipeline
# Ablation-driven / robust baseline style
#
# Main datasets:
# - dataloader_time_series / ts_kkt_id_okved_47.11_region_77_impute_True.csv
# - dataloader_time_series / ts_kkt_id_okved_47.19_region_77_impute_True.csv
# - dataloader_aggregated / agg_kkt_okved_47.11_region_77_impute_True.csv
# - dataloader_aggregated / agg_kkt_okved_47.19_region_77_impute_True.csv
#
# Outputs:
# - one TXT log per stage
# - one JSON/CSV per model run
# - ranked anomalous KKT devices
# - ranked anomalous KKT-day events
# - ablation comparison tables
# - explanation by top contributing features
# - final report
#
# Resume-safe:
# - existing result files are reused unless FORCE=True
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import math
import random
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Root with KKT data.
DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2")

# Output root near notebook.
OUT_ROOT = Path.cwd() / "kkt_anomaly_outputs"

LOG_DIR = OUT_ROOT / "logs"
JSON_DIR = OUT_ROOT / "results_json"
CSV_DIR = OUT_ROOT / "results_csv"
SUMMARY_DIR = OUT_ROOT / "summary"
CACHE_DIR = OUT_ROOT / "cache"

for d in [OUT_ROOT, LOG_DIR, JSON_DIR, CSV_DIR, SUMMARY_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FORCE = False

print("Python:", sys.executable)
print("Working dir:", Path.cwd())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)

if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)

Python: D:\Users\ivan\anaconda3\envs\gad_dgl\python.exe
Working dir: C:\Users\ivan\WORK\workshops\ICDM\Ablation
DATA_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2
DATA_ROOT exists: True
OUT_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs


In [9]:
# ============================================================
# Selected KKT files
# ============================================================

PATHS = {
    # Main time-series datasets.
    "ts_kkt_47_11": DATA_ROOT / "dataloader_time_series" / "ts_kkt_id_okved_47.11_region_77_impute_True.csv",
    "ts_kkt_47_19": DATA_ROOT / "dataloader_time_series" / "ts_kkt_id_okved_47.19_region_77_impute_True.csv",

    # Aggregated KKT-level feature datasets.
    "agg_kkt_47_11": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.11_region_77_impute_True.csv",
    "agg_kkt_47_19": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.19_region_77_impute_True.csv",

    # Raw DB outputs only for sanity check / optional linking.
    "raw_47_11": DATA_ROOT / "db_output" / "okved_47.11_region_77_2023_month_10-12_code_3_operation_1_v1.csv",
    "raw_47_19": DATA_ROOT / "db_output" / "okved_47.19_region_77_2023_month_10-12_code_3_operation_1_from_db.csv",
}

print("=" * 120)
print("PATH CHECK")
print("=" * 120)

for k, p in PATHS.items():
    status = "OK" if p.exists() else "MISSING"
    size = p.stat().st_size / 1024 / 1024 if p.exists() else None
    print(f"{status:<8} {k:<20} {str(p):<120} {'' if size is None else f'{size:.2f} MB'}")

PATH CHECK
OK       ts_kkt_47_11         C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.11_region_77_impute_True.csv 1024.72 MB
OK       ts_kkt_47_19         C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.19_region_77_impute_True.csv 451.60 MB
OK       agg_kkt_47_11        C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.11_region_77_impute_True.csv 78.62 MB
OK       agg_kkt_47_19        C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.19_region_77_impute_True.csv 35.49 MB
OK       raw_47_11            C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\db_output\okved_47.11_region_77_2023_month_10-12_code_3_operation_1_v1.csv 742.10 MB
OK       raw_47_19            C:\Users\ivan\WORK\workshops\ICDM\Ablation

In [10]:
# ============================================================
# Logging / saving helpers
# ============================================================

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def safe_name(x: str) -> str:
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def log_line(log_path: Path, msg: str, also_print: bool = True):
    line = f"[{now_str()}] {msg}"
    if also_print:
        print(line)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / 1024 / 1024


def save_df_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("[SAVED]", path, df.shape)

In [11]:
# ============================================================
# Logging / saving helpers
# ============================================================

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def safe_name(x: str) -> str:
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def log_line(log_path: Path, msg: str, also_print: bool = True):
    line = f"[{now_str()}] {msg}"
    if also_print:
        print(line)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / 1024 / 1024


def save_df_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("[SAVED]", path, df.shape)

In [13]:
# ============================================================
# Fast selected file audit
# ============================================================

audit_log = LOG_DIR / "00_selected_files_audit.txt"
audit_rows = []

for name, path in PATHS.items():
    if not path.exists():
        continue

    log_line(audit_log, "=" * 100)
    log_line(audit_log, f"FILE: {name}")
    log_line(audit_log, f"PATH: {path}")
    log_line(audit_log, f"SIZE MB: {file_size_mb(path):.2f}")

    try:
        sample = pd.read_csv(path, nrows=10)
        log_line(audit_log, f"SAMPLE SHAPE: {sample.shape}")
        log_line(audit_log, f"COLUMNS: {list(sample.columns)}")
        display(sample.head())

        audit_rows.append({
            "name": name,
            "path": str(path),
            "size_mb": round(file_size_mb(path), 3),
            "n_sample_cols": sample.shape[1],
            "columns": list(sample.columns),
        })

    except Exception as e:
        log_line(audit_log, f"READ ERROR: {repr(e)}")
        audit_rows.append({
            "name": name,
            "path": str(path),
            "size_mb": round(file_size_mb(path), 3),
            "error": repr(e),
        })

audit_df = pd.DataFrame(audit_rows)
save_df_csv(audit_df, SUMMARY_DIR / "selected_files_audit.csv")
display(audit_df)

[2026-05-17 19:49:36] ====================================================================================================
[2026-05-17 19:49:36] FILE: ts_kkt_47_11
[2026-05-17 19:49:36] PATH: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.11_region_77_impute_True.csv
[2026-05-17 19:49:36] SIZE MB: 1024.72
[2026-05-17 19:49:36] SAMPLE SHAPE: (10, 22)
[2026-05-17 19:49:36] COLUMNS: ['kkt_id', 'date', 'total_sum', 'avg_sum', 'std_sum', 'skew_sum', 'max_sum', 'min_sum', 'median_sum', 'total_cash', 'total_ecash', 'n_bills', 'n_items', 'total_items', 'trade_store_hash', 'trade_object_hash', 'ecash_fraction', 'avg_price_item', 'avg_price_piece', 'avg_cash', 'avg_ecash', 'ecash_bills_fraction']


,kkt_id,date,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,total_cash,...,n_items,total_items,trade_store_hash,trade_object_hash,ecash_fraction,avg_price_item,avg_price_piece,avg_cash,avg_ecash,ecash_bills_fraction
0,00079d6947ea9d8e7fb427f3d478c75e,2023-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
1,00079d6947ea9d8e7fb427f3d478c75e,2023-10-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
2,00079d6947ea9d8e7fb427f3d478c75e,2023-10-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
3,00079d6947ea9d8e7fb427f3d478c75e,2023-10-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0
4,00079d6947ea9d8e7fb427f3d478c75e,2023-10-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...,0.0,0.0,0.0,0.0,0.0,0.0


[2026-05-17 19:49:36] ====================================================================================================
[2026-05-17 19:49:36] FILE: ts_kkt_47_19
[2026-05-17 19:49:36] PATH: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.19_region_77_impute_True.csv
[2026-05-17 19:49:36] SIZE MB: 451.60
[2026-05-17 19:49:36] SAMPLE SHAPE: (10, 22)
[2026-05-17 19:49:36] COLUMNS: ['kkt_id', 'date', 'total_sum', 'avg_sum', 'std_sum', 'skew_sum', 'max_sum', 'min_sum', 'median_sum', 'total_cash', 'total_ecash', 'n_bills', 'n_items', 'total_items', 'trade_store_hash', 'trade_object_hash', 'ecash_fraction', 'avg_price_item', 'avg_price_piece', 'avg_cash', 'avg_ecash', 'ecash_bills_fraction']


,kkt_id,date,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,total_cash,...,n_items,total_items,trade_store_hash,trade_object_hash,ecash_fraction,avg_price_item,avg_price_piece,avg_cash,avg_ecash,ecash_bills_fraction
0,0000f85a36a983b9eb959950c1300e8b,2023-10-01,94603.85,2489.575000,2572.521296,2.110028,12345.38,191.98,1833.715,28129.0,...,643.0,733.414,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...,0.702665,147.128849,128.991061,2163.769231,2658.994000,0.657895
1,0000f85a36a983b9eb959950c1300e8b,2023-10-02,0.00,0.000000,0.000000,0.000000,0.00,0.00,0.000,0.0,...,0.0,0.000,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0000f85a36a983b9eb959950c1300e8b,2023-10-03,18893.18,944.659000,866.464034,1.060167,2880.13,29.95,639.575,10264.0,...,229.0,246.873,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...,0.456735,82.502969,76.529957,855.333333,1078.647500,0.400000
3,0000f85a36a983b9eb959950c1300e8b,2023-10-04,0.00,0.000000,0.000000,0.000000,0.00,0.00,0.000,0.0,...,0.0,0.000,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0000f85a36a983b9eb959950c1300e8b,2023-10-05,470646.86,1705.242246,3003.554822,6.002063,30855.00,15.00,888.230,126729.0,...,3776.0,5010.684,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...,0.730734,124.641647,93.928665,1195.556604,1987.964509,0.620072


[2026-05-17 19:49:36] ====================================================================================================
[2026-05-17 19:49:36] FILE: agg_kkt_47_11
[2026-05-17 19:49:36] PATH: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.11_region_77_impute_True.csv
[2026-05-17 19:49:36] SIZE MB: 78.62
[2026-05-17 19:49:36] SAMPLE SHAPE: (10, 156)
[2026-05-17 19:49:36] COLUMNS: ['avg_ecash__mean', 'avg_ecash__standard_deviation', 'avg_price_item__mean', 'avg_price_item__standard_deviation', 'avg_price_piece__mean', 'avg_price_piece__standard_deviation', 'avg_sum__mean', 'avg_sum__standard_deviation', 'avg_sum__skewness', 'avg_sum__ratio_beyond_r_sigma__r_2', 'avg_sum__ratio_beyond_r_sigma__r_3', 'ecash_bills_fraction__mean', 'ecash_bills_fraction__standard_deviation', 'ecash_bills_fraction__agg_linear_trend__attr_"slope"__chunk_len_7__f_agg_"mean"', 'ecash_bills_fraction__autocorrelation__lag_7', 'ecash_bills_fraction__

...

,kkt_id,date,region,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,...,n_ecash_bills,n_items,total_items,n_taxation_types,okved2,okved_group,type,category,trade_store_hash,trade_object_hash
0,00079d6947ea9d8e7fb427f3d478c75e,2023-12-08,77,3.00,3.000000,0.000000,NaN,3.00,3.00,3.000,...,0,1,1.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
1,00079d6947ea9d8e7fb427f3d478c75e,2023-12-15,77,33651.70,494.877941,580.649920,3.059434,3720.10,3.00,259.815,...,34,334,424.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
2,00079d6947ea9d8e7fb427f3d478c75e,2023-12-16,77,39556.65,648.469672,758.354135,1.939349,3586.93,3.00,344.000,...,40,307,409.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
3,00079d6947ea9d8e7fb427f3d478c75e,2023-12-17,77,38251.82,579.573030,546.617066,1.535047,2584.00,43.00,438.000,...,16,256,311.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...
4,00079d6947ea9d8e7fb427f3d478c75e,2023-12-18,77,70028.43,457.702157,549.172279,4.017840,4819.71,14.99,300.000,...,132,558,696.0,1,47.11,47.11,1,2,xae21a1bb91486cf5214c453fa06fac72394c8b5c6b0e3...,x4583ce7af75ad98b25c87d6e0d69807cbc0e7badee6d9...


[2026-05-17 19:49:36] ====================================================================================================
[2026-05-17 19:49:36] FILE: raw_47_19
[2026-05-17 19:49:36] PATH: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\db_output\okved_47.19_region_77_2023_month_10-12_code_3_operation_1_from_db.csv
[2026-05-17 19:49:36] SIZE MB: 321.63
[2026-05-17 19:49:36] SAMPLE SHAPE: (10, 27)
[2026-05-17 19:49:36] COLUMNS: ['kkt_id', 'date', 'region', 'total_sum', 'avg_sum', 'std_sum', 'skew_sum', 'max_sum', 'min_sum', 'median_sum', 'total_cash', 'total_ecash', 'total_prepaid', 'total_credit', 'total_provision', 'n_bills', 'n_cash_bills', 'n_ecash_bills', 'n_items', 'total_items', 'n_taxation_types', 'okved2', 'okved_group', 'type', 'category', 'trade_store_hash', 'trade_object_hash']


,kkt_id,date,region,total_sum,avg_sum,std_sum,skew_sum,max_sum,min_sum,median_sum,...,n_ecash_bills,n_items,total_items,n_taxation_types,okved2,okved_group,type,category,trade_store_hash,trade_object_hash
0,0000f85a36a983b9eb959950c1300e8b,2023-10-01,77,94603.85,2489.575000,2572.521296,2.110028,12345.38,191.98,1833.715,...,25,643,733.414,1,47.19,47.19,1,4,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...
1,0000f85a36a983b9eb959950c1300e8b,2023-10-03,77,18893.18,944.659000,866.464034,1.060167,2880.13,29.95,639.575,...,8,229,246.873,1,47.19,47.19,1,4,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...
2,0000f85a36a983b9eb959950c1300e8b,2023-10-05,77,470646.86,1705.242246,3003.554822,6.002063,30855.00,15.00,888.230,...,173,3776,5010.684,1,47.19,47.19,1,4,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...
3,0000f85a36a983b9eb959950c1300e8b,2023-10-06,77,469380.62,1713.067956,1804.771119,1.866532,11488.41,7.00,1100.185,...,191,3747,4608.293,1,47.19,47.19,1,4,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...
4,0000f85a36a983b9eb959950c1300e8b,2023-10-07,77,625890.80,2361.852075,2681.756082,2.567190,19998.40,1.05,1494.000,...,197,4620,5287.014,1,47.19,47.19,1,4,x049f57ea911ddd05f02837ade7ed93a05ae181d83aafc...,xbfeeb8c9a053f91fca63d1cbfac00d3e104667cdde102...


[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\selected_files_audit.csv (6, 5)


,name,path,size_mb,n_sample_cols,columns
0,ts_kkt_47_11,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,1024.720,22,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
1,ts_kkt_47_19,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,451.602,22,"[kkt_id, date, total_sum, avg_sum, std_sum, sk..."
2,agg_kkt_47_11,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,78.621,156,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
3,agg_kkt_47_19,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,35.495,156,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
4,raw_47_11,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,742.099,27,"[kkt_id, date, region, total_sum, avg_sum, std..."
5,raw_47_19,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,321.633,27,"[kkt_id, date, region, total_sum, avg_sum, std..."


In [12]:
# ============================================================
# Feature groups for ablation
# ============================================================

ID_COLS = [
    "kkt_id",
    "trade_store_hash",
    "trade_object_hash",
    "region",
    "okved2",
    "okved_group",
    "type",
    "category",
    "date",
]

def infer_feature_group(col: str) -> str:
    c = str(col).lower()

    if c in ["kkt_id", "trade_store_hash", "trade_object_hash", "region", "okved2", "okved_group", "type", "category", "date"]:
        return "id"

    # Aggregate feature names usually look like total_sum__mean, n_bills__skewness, etc.
    if "total_sum" in c or "avg_sum" in c or "max_sum" in c or "min_sum" in c or "median_sum" in c or "std_sum" in c or "skew_sum" in c:
        return "turnover"

    if "bill" in c or "n_bills" in c:
        return "bills"

    if "item" in c or "price" in c or "piece" in c:
        return "items_price"

    if "cash" in c or "ecash" in c or "payment" in c:
        return "payment_mix"

    if "autocorrelation" in c or "partial_autocorrelation" in c or "trend" in c or "seasonal" in c or "frequency" in c or "fft" in c or "phase" in c:
        return "temporal"

    if "benford" in c:
        return "benford"

    if "corr" in c:
        return "cross_corr"

    if "ratio_beyond" in c or "number_peaks" in c or "longest_strike" in c or "cid_ce" in c or "mean_abs_change" in c:
        return "shape_complexity"

    return "other_numeric"


FEATURE_GROUPS = [
    "all_numeric",
    "turnover",
    "bills",
    "items_price",
    "payment_mix",
    "temporal",
    "benford",
    "cross_corr",
    "shape_complexity",
    "non_temporal",
    "simple_core",
]


SIMPLE_CORE_COL_HINTS = [
    "total_sum",
    "avg_sum",
    "max_sum",
    "min_sum",
    "median_sum",
    "std_sum",
    "total_cash",
    "total_ecash",
    "n_bills",
    "n_items",
    "total_items",
    "ecash_fraction",
    "cash_fraction",
    "avg_price_item",
    "avg_price_piece",
    "avg_cash",
    "avg_ecash",
]


def get_numeric_feature_columns(df: pd.DataFrame) -> List[str]:
    cols = []
    for c in df.columns:
        if c in ID_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def select_columns_by_group(df: pd.DataFrame, group: str) -> List[str]:
    numeric_cols = get_numeric_feature_columns(df)

    if group == "all_numeric":
        return numeric_cols

    if group == "non_temporal":
        return [c for c in numeric_cols if infer_feature_group(c) not in ["temporal"]]

    if group == "simple_core":
        selected = []
        for c in numeric_cols:
            cl = str(c).lower()
            if any(h in cl for h in SIMPLE_CORE_COL_HINTS):
                selected.append(c)
        return selected

    return [c for c in numeric_cols if infer_feature_group(c) == group]


def describe_feature_groups(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group in FEATURE_GROUPS:
        cols = select_columns_by_group(df, group)
        rows.append({
            "group": group,
            "n_cols": len(cols),
            "columns_preview": cols[:20],
        })
    return pd.DataFrame(rows)

In [13]:
# ============================================================
# Unsupervised anomaly scoring models
# ============================================================

def prepare_matrix(
    df: pd.DataFrame,
    feature_cols: List[str],
    scaler: str = "robust",
) -> Tuple[np.ndarray, Pipeline]:
    X = df[feature_cols].copy()

    # Replace inf.
    X = X.replace([np.inf, -np.inf], np.nan)

    if scaler == "robust":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ])
    elif scaler == "standard":
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])

    X_scaled = pipe.fit_transform(X)
    X_scaled = np.asarray(X_scaled, dtype=np.float32)

    # Safety.
    X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

    return X_scaled, pipe


def normalize_score(s: np.ndarray) -> np.ndarray:
    s = np.asarray(s, dtype=float)
    s = np.nan_to_num(s, nan=np.nanmedian(s), posinf=np.nanmax(s[np.isfinite(s)]) if np.isfinite(s).any() else 0, neginf=np.nanmin(s[np.isfinite(s)]) if np.isfinite(s).any() else 0)
    lo, hi = np.percentile(s, [1, 99])
    if hi <= lo:
        lo, hi = np.min(s), np.max(s)
    if hi <= lo:
        return np.zeros_like(s, dtype=float)
    z = (s - lo) / (hi - lo)
    return np.clip(z, 0, 1)


def robust_z_score_anomaly(X_scaled: np.ndarray) -> np.ndarray:
    # After robust scaling, large absolute values are suspicious.
    score = np.sqrt(np.mean(np.square(X_scaled), axis=1))
    return normalize_score(score)


def pca_reconstruction_anomaly(X_scaled: np.ndarray, n_components: Optional[int] = None) -> np.ndarray:
    n, d = X_scaled.shape
    if d <= 1:
        return robust_z_score_anomaly(X_scaled)

    if n_components is None:
        n_components = min(10, max(1, d // 3), d - 1, n - 1)

    if n_components < 1:
        return robust_z_score_anomaly(X_scaled)

    pca = PCA(n_components=n_components, random_state=SEED)
    Z = pca.fit_transform(X_scaled)
    X_rec = pca.inverse_transform(Z)
    err = np.mean((X_scaled - X_rec) ** 2, axis=1)
    return normalize_score(err)


def isolation_forest_anomaly(X_scaled: np.ndarray, contamination: float = 0.03) -> np.ndarray:
    model = IsolationForest(
        n_estimators=400,
        contamination=contamination,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_scaled)
    # Lower decision_function = more abnormal.
    score = -model.decision_function(X_scaled)
    return normalize_score(score)


def lof_anomaly(X_scaled: np.ndarray, contamination: float = 0.03, max_n: int = 120_000) -> Optional[np.ndarray]:
    n = X_scaled.shape[0]
    if n > max_n:
        return None

    model = LocalOutlierFactor(
        n_neighbors=35,
        contamination=contamination,
        novelty=False,
        n_jobs=-1,
    )
    pred = model.fit_predict(X_scaled)
    score = -model.negative_outlier_factor_
    return normalize_score(score)


def ocsvm_anomaly(X_scaled: np.ndarray, contamination: float = 0.03, max_n: int = 30_000) -> Optional[np.ndarray]:
    n = X_scaled.shape[0]
    if n > max_n:
        return None

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=contamination,
    )
    model.fit(X_scaled)
    score = -model.decision_function(X_scaled)
    return normalize_score(score)


def elliptic_anomaly(X_scaled: np.ndarray, contamination: float = 0.03, max_n: int = 100_000, max_d: int = 80) -> Optional[np.ndarray]:
    n, d = X_scaled.shape
    if n > max_n or d > max_d or n <= d + 10:
        return None

    model = EllipticEnvelope(
        contamination=contamination,
        random_state=SEED,
        support_fraction=None,
    )
    model.fit(X_scaled)
    score = -model.decision_function(X_scaled)
    return normalize_score(score)


def compute_anomaly_scores(
    X_scaled: np.ndarray,
    contamination: float = 0.03,
    allow_heavy: bool = False,
) -> Dict[str, np.ndarray]:
    scores = {}

    scores["robust_z"] = robust_z_score_anomaly(X_scaled)
    scores["pca_reconstruction"] = pca_reconstruction_anomaly(X_scaled)
    scores["isolation_forest"] = isolation_forest_anomaly(X_scaled, contamination=contamination)

    lof_score = lof_anomaly(X_scaled, contamination=contamination)
    if lof_score is not None:
        scores["lof"] = lof_score

    if allow_heavy:
        ocsvm_score = ocsvm_anomaly(X_scaled, contamination=contamination)
        if ocsvm_score is not None:
            scores["one_class_svm"] = ocsvm_score

    elliptic_score = elliptic_anomaly(X_scaled, contamination=contamination)
    if elliptic_score is not None:
        scores["elliptic_envelope"] = elliptic_score

    # Consensus.
    score_mat = np.vstack([normalize_score(v) for v in scores.values()]).T
    scores["consensus_mean"] = np.mean(score_mat, axis=1)
    scores["consensus_max"] = np.max(score_mat, axis=1)

    return scores

In [14]:
# ============================================================
# Feature contribution explanations
# ============================================================

def top_feature_contributions(
    X_scaled: np.ndarray,
    feature_cols: List[str],
    row_idx: int,
    top_n: int = 10,
) -> List[Dict[str, Any]]:
    row = X_scaled[row_idx]
    abs_vals = np.abs(row)
    order = np.argsort(-abs_vals)[:top_n]

    out = []
    for j in order:
        out.append({
            "feature": feature_cols[j],
            "scaled_value": float(row[j]),
            "abs_scaled_value": float(abs_vals[j]),
            "feature_group": infer_feature_group(feature_cols[j]),
        })

    return out


def add_explanations_to_top(
    top_df: pd.DataFrame,
    X_scaled: np.ndarray,
    feature_cols: List[str],
    original_index_col: str = "_row_id",
    top_n_features: int = 8,
) -> pd.DataFrame:
    explanations = []

    for _, r in top_df.iterrows():
        idx = int(r[original_index_col])
        contrib = top_feature_contributions(X_scaled, feature_cols, idx, top_n=top_n_features)
        expl = "; ".join([
            f"{c['feature']}={c['scaled_value']:.2f}"
            for c in contrib
        ])
        groups = "; ".join([
            f"{c['feature_group']}:{c['feature']}"
            for c in contrib
        ])
        explanations.append({
            "top_feature_explanation": expl,
            "top_feature_groups": groups,
        })

    expl_df = pd.DataFrame(explanations)
    return pd.concat([top_df.reset_index(drop=True), expl_df], axis=1)

In [15]:
# ============================================================
# Aggregated KKT-level anomaly detection
# ============================================================

AGG_DATASETS = {
    "agg_kkt_47_11": {
        "path": PATHS["agg_kkt_47_11"],
        "okved_group": "47.11",
    },
    "agg_kkt_47_19": {
        "path": PATHS["agg_kkt_47_19"],
        "okved_group": "47.19",
    },
}

CONTAMINATION_LEVELS = [0.01, 0.03, 0.05]

def load_agg_dataset(name: str, path: Path) -> pd.DataFrame:
    log_path = LOG_DIR / f"load_{safe_name(name)}.txt"

    log_line(log_path, f"Loading aggregated KKT dataset: {name}")
    log_line(log_path, f"Path: {path}")
    log_line(log_path, f"Size MB: {file_size_mb(path):.2f}")

    df = pd.read_csv(path)
    df["_row_id"] = np.arange(len(df))
    df["_source_dataset"] = name

    log_line(log_path, f"Shape: {df.shape}")
    log_line(log_path, f"Columns: {len(df.columns)}")
    log_line(log_path, f"First columns: {list(df.columns[:30])}")

    return df


def run_agg_ablation_experiment(
    dataset_name: str,
    df: pd.DataFrame,
    group: str,
    contamination: float,
    force: bool = FORCE,
) -> Dict[str, Any]:

    run_id = f"{dataset_name}__{group}__contam_{contamination}"
    json_path = JSON_DIR / f"agg__{safe_name(run_id)}.json"
    csv_path = CSV_DIR / f"agg__{safe_name(run_id)}__ranked.csv"
    top_csv_path = CSV_DIR / f"agg__{safe_name(run_id)}__top_anomalies.csv"
    txt_path = LOG_DIR / f"agg__{safe_name(run_id)}.txt"

    if json_path.exists() and csv_path.exists() and not force:
        log_line(txt_path, f"[SKIP] existing run: {run_id}")
        return load_json(json_path)

    t0 = time.time()

    log_line(txt_path, "=" * 120)
    log_line(txt_path, f"RUN AGG dataset={dataset_name}, group={group}, contamination={contamination}")
    log_line(txt_path, "=" * 120)

    feature_cols = select_columns_by_group(df, group)

    if len(feature_cols) == 0:
        result = {
            "status": "skipped_no_features",
            "dataset": dataset_name,
            "group": group,
            "contamination": contamination,
            "n_rows": len(df),
            "n_features": 0,
        }
        save_json(json_path, result)
        return result

    log_line(txt_path, f"n_rows: {len(df)}")
    log_line(txt_path, f"n_features: {len(feature_cols)}")
    log_line(txt_path, f"feature preview: {feature_cols[:30]}")

    X_scaled, prep = prepare_matrix(df, feature_cols, scaler="robust")

    scores = compute_anomaly_scores(
        X_scaled,
        contamination=contamination,
        allow_heavy=False,
    )

    out = df.copy()

    for model_name, s in scores.items():
        out[f"score_{model_name}"] = normalize_score(s)

    # Main score: consensus_mean.
    out["anomaly_score"] = out["score_consensus_mean"]
    out["anomaly_rank"] = out["anomaly_score"].rank(ascending=False, method="first").astype(int)
    out["anomaly_percentile"] = out["anomaly_score"].rank(pct=True)

    threshold = np.quantile(out["anomaly_score"], 1.0 - contamination)
    out["is_top_anomaly"] = out["anomaly_score"] >= threshold

    out = out.sort_values("anomaly_score", ascending=False).reset_index(drop=True)

    # Top rows with explanations.
    top_n = min(500, max(50, int(len(out) * max(contamination, 0.01))))
    top_df = out.head(top_n).copy()
    top_df = add_explanations_to_top(
        top_df=top_df,
        X_scaled=X_scaled,
        feature_cols=feature_cols,
        original_index_col="_row_id",
        top_n_features=10,
    )

    save_df_csv(out, csv_path)
    save_df_csv(top_df, top_csv_path)

    runtime = time.time() - t0

    result = {
        "status": "ok",
        "stage": "agg_kkt_level",
        "dataset": dataset_name,
        "group": group,
        "contamination": contamination,
        "n_rows": int(len(df)),
        "n_features": int(len(feature_cols)),
        "feature_cols": feature_cols,
        "score_columns": list(scores.keys()),
        "threshold": float(threshold),
        "n_top_anomalies": int(out["is_top_anomaly"].sum()),
        "top_score": float(out["anomaly_score"].max()),
        "median_score": float(out["anomaly_score"].median()),
        "runtime_sec": float(runtime),
        "ranked_csv": str(csv_path),
        "top_csv": str(top_csv_path),
        "created_at": now_str(),
    }

    save_json(json_path, result)

    log_line(txt_path, f"threshold: {threshold:.6f}")
    log_line(txt_path, f"n_top_anomalies: {int(out['is_top_anomaly'].sum())}")
    log_line(txt_path, f"top_score: {out['anomaly_score'].max():.6f}")
    log_line(txt_path, f"Saved ranked: {csv_path}")
    log_line(txt_path, f"Saved top: {top_csv_path}")
    log_line(txt_path, f"Runtime sec: {runtime:.2f}")

    return result


agg_results = []

for dataset_name, meta in AGG_DATASETS.items():
    path = meta["path"]
    if not path.exists():
        print("[MISSING]", path)
        continue

    df_agg = load_agg_dataset(dataset_name, path)

    print("=" * 120)
    print("Feature groups:", dataset_name)
    print("=" * 120)
    display(describe_feature_groups(df_agg))

    for group in FEATURE_GROUPS:
        for contamination in CONTAMINATION_LEVELS:
            print("\n" + "=" * 120)
            print(f"AGG RUN dataset={dataset_name}, group={group}, contamination={contamination}")
            print("=" * 120)

            res = run_agg_ablation_experiment(
                dataset_name=dataset_name,
                df=df_agg,
                group=group,
                contamination=contamination,
                force=FORCE,
            )
            agg_results.append(res)

[2026-05-18 07:50:47] Loading aggregated KKT dataset: agg_kkt_47_11
[2026-05-18 07:50:47] Path: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.11_region_77_impute_True.csv
[2026-05-18 07:50:47] Size MB: 78.62
[2026-05-18 07:50:49] Shape: (34443, 158)
[2026-05-18 07:50:49] Columns: 158
[2026-05-18 07:50:49] First columns: ['avg_ecash__mean', 'avg_ecash__standard_deviation', 'avg_price_item__mean', 'avg_price_item__standard_deviation', 'avg_price_piece__mean', 'avg_price_piece__standard_deviation', 'avg_sum__mean', 'avg_sum__standard_deviation', 'avg_sum__skewness', 'avg_sum__ratio_beyond_r_sigma__r_2', 'avg_sum__ratio_beyond_r_sigma__r_3', 'ecash_bills_fraction__mean', 'ecash_bills_fraction__standard_deviation', 'ecash_bills_fraction__agg_linear_trend__attr_"slope"__chunk_len_7__f_agg_"mean"', 'ecash_bills_fraction__autocorrelation__lag_7', 'ecash_bills_fraction__ratio_beyond_r_sigma__r_2', 'ecash_fraction__mean', 'ecash_f

,group,n_cols,columns_preview
0,all_numeric,150,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
1,turnover,107,"[avg_sum__mean, avg_sum__standard_deviation, a..."
2,bills,11,"[ecash_bills_fraction__mean, ecash_bills_fract..."
3,items_price,8,"[avg_price_item__mean, avg_price_item__standar..."
4,payment_mix,22,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
5,temporal,0,[]
6,benford,0,[]
7,cross_corr,0,[]
8,shape_complexity,0,[]
9,non_temporal,150,"[avg_ecash__mean, avg_ecash__standard_deviatio..."



AGG RUN dataset=agg_kkt_47_11, group=all_numeric, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__all_numeric__contam_0.01

AGG RUN dataset=agg_kkt_47_11, group=all_numeric, contamination=0.03
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__all_numeric__contam_0.03

AGG RUN dataset=agg_kkt_47_11, group=all_numeric, contamination=0.05
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__all_numeric__contam_0.05

AGG RUN dataset=agg_kkt_47_11, group=turnover, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__turnover__contam_0.01

AGG RUN dataset=agg_kkt_47_11, group=turnover, contamination=0.03
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__turnover__contam_0.03

AGG RUN dataset=agg_kkt_47_11, group=turnover, contamination=0.05
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_11__turnover__contam_0.05

AGG RUN dataset=agg_kkt_47_11, group=bills, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existin

,group,n_cols,columns_preview
0,all_numeric,150,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
1,turnover,107,"[avg_sum__mean, avg_sum__standard_deviation, a..."
2,bills,11,"[ecash_bills_fraction__mean, ecash_bills_fract..."
3,items_price,8,"[avg_price_item__mean, avg_price_item__standar..."
4,payment_mix,22,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
5,temporal,0,[]
6,benford,0,[]
7,cross_corr,0,[]
8,shape_complexity,0,[]
9,non_temporal,150,"[avg_ecash__mean, avg_ecash__standard_deviatio..."



AGG RUN dataset=agg_kkt_47_19, group=all_numeric, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__all_numeric__contam_0.01

AGG RUN dataset=agg_kkt_47_19, group=all_numeric, contamination=0.03
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__all_numeric__contam_0.03

AGG RUN dataset=agg_kkt_47_19, group=all_numeric, contamination=0.05
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__all_numeric__contam_0.05

AGG RUN dataset=agg_kkt_47_19, group=turnover, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__turnover__contam_0.01

AGG RUN dataset=agg_kkt_47_19, group=turnover, contamination=0.03
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__turnover__contam_0.03

AGG RUN dataset=agg_kkt_47_19, group=turnover, contamination=0.05
[2026-05-18 07:50:49] [SKIP] existing run: agg_kkt_47_19__turnover__contam_0.05

AGG RUN dataset=agg_kkt_47_19, group=bills, contamination=0.01
[2026-05-18 07:50:49] [SKIP] existin

In [18]:
# ============================================================
# Time-series KKT-day anomaly detection
# ============================================================

TS_DATASETS = {
    "ts_kkt_47_11": {
        "path": PATHS["ts_kkt_47_11"],
        "okved_group": "47.11",
    },
    "ts_kkt_47_19": {
        "path": PATHS["ts_kkt_47_19"],
        "okved_group": "47.19",
    },
}

TS_NUMERIC_COLS = [
    "total_sum",
    "avg_sum",
    "std_sum",
    "skew_sum",
    "max_sum",
    "min_sum",
    "median_sum",
    "total_cash",
    "total_ecash",
    "n_bills",
    "n_items",
    "total_items",
    "ecash_fraction",
    "avg_price_item",
    "avg_price_piece",
    "avg_cash",
    "avg_ecash",
    "ecash_bills_fraction",
]

TS_ID_COLS = [
    "kkt_id",
    "date",
    "trade_store_hash",
    "trade_object_hash",
]


def load_ts_dataset(name: str, path: Path, max_rows: Optional[int] = None) -> pd.DataFrame:
    log_path = LOG_DIR / f"load_{safe_name(name)}.txt"

    log_line(log_path, f"Loading time-series KKT dataset: {name}")
    log_line(log_path, f"Path: {path}")
    log_line(log_path, f"Size MB: {file_size_mb(path):.2f}")

    sample = pd.read_csv(path, nrows=5)
    available_cols = list(sample.columns)

    usecols = [c for c in TS_ID_COLS + TS_NUMERIC_COLS if c in available_cols]

    log_line(log_path, f"Usecols: {usecols}")

    df = pd.read_csv(path, usecols=usecols, nrows=max_rows)
    df["_row_id"] = np.arange(len(df))
    df["_source_dataset"] = name

    # Parse date.
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Downcast numeric columns.
    for c in TS_NUMERIC_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

    log_line(log_path, f"Shape: {df.shape}")
    log_line(log_path, f"Date range: {df['date'].min()} -> {df['date'].max()}")
    log_line(log_path, f"Unique kkt_id: {df['kkt_id'].nunique() if 'kkt_id' in df.columns else 'NA'}")

    return df


def add_ts_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    eps = 1e-9

    if "total_sum" in out.columns and "n_bills" in out.columns:
        out["derived_sum_per_bill"] = out["total_sum"] / (out["n_bills"] + eps)

    if "total_items" in out.columns and "n_bills" in out.columns:
        out["derived_items_per_bill"] = out["total_items"] / (out["n_bills"] + eps)

    if "total_sum" in out.columns and "total_items" in out.columns:
        out["derived_sum_per_item"] = out["total_sum"] / (out["total_items"] + eps)

    if "total_ecash" in out.columns and "total_sum" in out.columns:
        out["derived_ecash_share"] = out["total_ecash"] / (out["total_sum"] + eps)

    if "total_cash" in out.columns and "total_sum" in out.columns:
        out["derived_cash_share"] = out["total_cash"] / (out["total_sum"] + eps)

    if "total_cash" in out.columns and "total_ecash" in out.columns:
        out["derived_cash_to_ecash"] = out["total_cash"] / (out["total_ecash"] + eps)

    # log transforms for heavy-tailed values.
    for c in [
        "total_sum",
        "avg_sum",
        "max_sum",
        "median_sum",
        "total_cash",
        "total_ecash",
        "n_bills",
        "n_items",
        "total_items",
    ]:
        if c in out.columns:
            out[f"log1p_{c}"] = np.log1p(np.maximum(out[c].astype(float), 0.0))

    return out


def add_groupwise_rolling_features(
    df: pd.DataFrame,
    group_col: str = "kkt_id",
    date_col: str = "date",
    base_cols: Optional[List[str]] = None,
    windows: List[int] = [7, 14],
) -> pd.DataFrame:
    out = df.copy()

    if base_cols is None:
        base_cols = [
            "total_sum",
            "n_bills",
            "total_items",
            "ecash_fraction",
            "avg_sum",
            "max_sum",
        ]

    base_cols = [c for c in base_cols if c in out.columns]

    out = out.sort_values([group_col, date_col]).reset_index(drop=True)

    for c in base_cols:
        for w in windows:
            grp = out.groupby(group_col, sort=False)[c]

            roll_mean = grp.transform(lambda s: s.shift(1).rolling(w, min_periods=3).mean())
            roll_std = grp.transform(lambda s: s.shift(1).rolling(w, min_periods=3).std())

            out[f"{c}__roll{w}_mean"] = roll_mean.astype("float32")
            out[f"{c}__roll{w}_std"] = roll_std.astype("float32")
            out[f"{c}__roll{w}_z"] = ((out[c] - roll_mean) / (roll_std + 1e-6)).astype("float32")

            # Relative jump.
            out[f"{c}__roll{w}_rel_jump"] = ((out[c] - roll_mean) / (np.abs(roll_mean) + 1e-6)).astype("float32")

    return out


def add_peer_date_features(
    df: pd.DataFrame,
    date_col: str = "date",
    base_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    out = df.copy()

    if base_cols is None:
        base_cols = [
            "total_sum",
            "n_bills",
            "total_items",
            "ecash_fraction",
            "avg_sum",
            "max_sum",
        ]

    base_cols = [c for c in base_cols if c in out.columns]

    for c in base_cols:
        med = out.groupby(date_col)[c].transform("median")
        q1 = out.groupby(date_col)[c].transform(lambda s: s.quantile(0.25))
        q3 = out.groupby(date_col)[c].transform(lambda s: s.quantile(0.75))
        iqr = q3 - q1

        out[f"{c}__peer_date_robust_z"] = ((out[c] - med) / (iqr + 1e-6)).astype("float32")

    return out


def build_ts_features(name: str, path: Path, max_rows: Optional[int] = None, force: bool = FORCE) -> pd.DataFrame:
    cache_path = CACHE_DIR / f"{safe_name(name)}__ts_features.parquet"
    fallback_csv = CACHE_DIR / f"{safe_name(name)}__ts_features.csv"

    log_path = LOG_DIR / f"build_ts_features_{safe_name(name)}.txt"

    if cache_path.exists() and not force:
        log_line(log_path, f"[SKIP] Loading cached parquet: {cache_path}")
        return pd.read_parquet(cache_path)

    if fallback_csv.exists() and not force:
        log_line(log_path, f"[SKIP] Loading cached csv: {fallback_csv}")
        return pd.read_csv(fallback_csv, parse_dates=["date"])

    t0 = time.time()

    log_line(log_path, "=" * 120)
    log_line(log_path, f"Building time-series features for {name}")
    log_line(log_path, "=" * 120)

    df = load_ts_dataset(name, path, max_rows=max_rows)
    log_line(log_path, f"Loaded shape: {df.shape}")

    df = add_ts_derived_features(df)
    log_line(log_path, f"After derived features: {df.shape}")

    df = add_groupwise_rolling_features(df)
    log_line(log_path, f"After rolling features: {df.shape}")

    df = add_peer_date_features(df)
    log_line(log_path, f"After peer-date features: {df.shape}")

    # Save cache.
    try:
        df.to_parquet(cache_path, index=False)
        log_line(log_path, f"Saved parquet cache: {cache_path}")
    except Exception as e:
        log_line(log_path, f"[WARN] parquet save failed: {repr(e)}")
        df.to_csv(fallback_csv, index=False, encoding="utf-8-sig")
        log_line(log_path, f"Saved CSV cache: {fallback_csv}")

    log_line(log_path, f"Runtime sec: {time.time() - t0:.2f}")

    return df


TS_FEATURE_GROUPS = [
    "all_numeric",
    "core_daily",
    "rolling_deviation",
    "peer_deviation",
    "payment_mix",
    "turnover",
    "bills_items",
]


def select_ts_columns_by_group(df: pd.DataFrame, group: str) -> List[str]:
    numeric_cols = []
    for c in df.columns:
        if c in ["_row_id"] or c in TS_ID_COLS or c == "_source_dataset":
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)

    if group == "all_numeric":
        return numeric_cols

    if group == "core_daily":
        return [c for c in numeric_cols if "__" not in c and not c.startswith("derived_") and not c.startswith("log1p_")]

    if group == "rolling_deviation":
        return [c for c in numeric_cols if "roll" in c]

    if group == "peer_deviation":
        return [c for c in numeric_cols if "peer_date" in c]

    if group == "payment_mix":
        return [c for c in numeric_cols if "cash" in c.lower() or "ecash" in c.lower()]

    if group == "turnover":
        return [c for c in numeric_cols if "sum" in c.lower() or "price" in c.lower()]

    if group == "bills_items":
        return [c for c in numeric_cols if "bill" in c.lower() or "item" in c.lower()]

    raise ValueError(group)

In [19]:
# ============================================================
# Run KKT-day anomaly detection
# ============================================================

def run_ts_anomaly_experiment(
    dataset_name: str,
    df: pd.DataFrame,
    group: str,
    contamination: float,
    force: bool = FORCE,
) -> Dict[str, Any]:

    run_id = f"{dataset_name}__{group}__contam_{contamination}"
    json_path = JSON_DIR / f"ts__{safe_name(run_id)}.json"
    csv_path = CSV_DIR / f"ts__{safe_name(run_id)}__ranked_kkt_days.csv"
    top_csv_path = CSV_DIR / f"ts__{safe_name(run_id)}__top_kkt_days.csv"
    device_csv_path = CSV_DIR / f"ts__{safe_name(run_id)}__device_aggregated_scores.csv"
    txt_path = LOG_DIR / f"ts__{safe_name(run_id)}.txt"

    if json_path.exists() and csv_path.exists() and not force:
        log_line(txt_path, f"[SKIP] existing run: {run_id}")
        return load_json(json_path)

    t0 = time.time()

    log_line(txt_path, "=" * 120)
    log_line(txt_path, f"RUN TS dataset={dataset_name}, group={group}, contamination={contamination}")
    log_line(txt_path, "=" * 120)

    feature_cols = select_ts_columns_by_group(df, group)

    if len(feature_cols) == 0:
        result = {
            "status": "skipped_no_features",
            "dataset": dataset_name,
            "group": group,
            "contamination": contamination,
            "n_rows": len(df),
            "n_features": 0,
        }
        save_json(json_path, result)
        return result

    log_line(txt_path, f"n_rows: {len(df)}")
    log_line(txt_path, f"n_features: {len(feature_cols)}")
    log_line(txt_path, f"feature preview: {feature_cols[:30]}")

    X_scaled, prep = prepare_matrix(df, feature_cols, scaler="robust")

    scores = compute_anomaly_scores(
        X_scaled,
        contamination=contamination,
        allow_heavy=False,
    )

    out_cols = [c for c in ["kkt_id", "date", "trade_store_hash", "trade_object_hash", "_row_id", "_source_dataset"] if c in df.columns]
    out = df[out_cols].copy()

    # Add selected original interpretable columns.
    for c in [
        "total_sum",
        "avg_sum",
        "max_sum",
        "min_sum",
        "median_sum",
        "total_cash",
        "total_ecash",
        "n_bills",
        "n_items",
        "total_items",
        "ecash_fraction",
        "avg_price_item",
        "avg_price_piece",
        "derived_sum_per_bill",
        "derived_items_per_bill",
        "derived_sum_per_item",
        "derived_ecash_share",
    ]:
        if c in df.columns:
            out[c] = df[c]

    for model_name, s in scores.items():
        out[f"score_{model_name}"] = normalize_score(s)

    out["anomaly_score"] = out["score_consensus_mean"]
    out["anomaly_rank"] = out["anomaly_score"].rank(ascending=False, method="first").astype(int)

    threshold = np.quantile(out["anomaly_score"], 1.0 - contamination)
    out["is_top_anomaly"] = out["anomaly_score"] >= threshold

    out = out.sort_values("anomaly_score", ascending=False).reset_index(drop=True)

    top_n = min(5000, max(100, int(len(out) * max(contamination, 0.005))))
    top_df = out.head(top_n).copy()
    top_df = add_explanations_to_top(
        top_df=top_df,
        X_scaled=X_scaled,
        feature_cols=feature_cols,
        original_index_col="_row_id",
        top_n_features=10,
    )

    # Aggregate KKT-level score from daily anomalies.
    device_agg = (
        out
        .groupby("kkt_id", as_index=False)
        .agg(
            device_score_mean=("anomaly_score", "mean"),
            device_score_max=("anomaly_score", "max"),
            device_score_p95=("anomaly_score", lambda s: float(np.quantile(s, 0.95))),
            n_days=("anomaly_score", "size"),
            n_top_anomaly_days=("is_top_anomaly", "sum"),
            total_sum_sum=("total_sum", "sum") if "total_sum" in out.columns else ("anomaly_score", "size"),
            n_bills_sum=("n_bills", "sum") if "n_bills" in out.columns else ("anomaly_score", "size"),
        )
    )

    device_agg["device_score_combined"] = normalize_score(
        0.50 * normalize_score(device_agg["device_score_max"].to_numpy())
        + 0.30 * normalize_score(device_agg["device_score_p95"].to_numpy())
        + 0.20 * normalize_score(device_agg["n_top_anomaly_days"].to_numpy())
    )

    device_agg = device_agg.sort_values("device_score_combined", ascending=False).reset_index(drop=True)
    device_agg["device_anomaly_rank"] = np.arange(1, len(device_agg) + 1)

    save_df_csv(out, csv_path)
    save_df_csv(top_df, top_csv_path)
    save_df_csv(device_agg, device_csv_path)

    runtime = time.time() - t0

    result = {
        "status": "ok",
        "stage": "ts_kkt_day_level",
        "dataset": dataset_name,
        "group": group,
        "contamination": contamination,
        "n_rows": int(len(df)),
        "n_devices": int(df["kkt_id"].nunique()),
        "n_features": int(len(feature_cols)),
        "feature_cols": feature_cols,
        "score_columns": list(scores.keys()),
        "threshold": float(threshold),
        "n_top_anomaly_days": int(out["is_top_anomaly"].sum()),
        "top_score": float(out["anomaly_score"].max()),
        "median_score": float(out["anomaly_score"].median()),
        "runtime_sec": float(runtime),
        "ranked_csv": str(csv_path),
        "top_csv": str(top_csv_path),
        "device_csv": str(device_csv_path),
        "created_at": now_str(),
    }

    save_json(json_path, result)

    log_line(txt_path, f"threshold: {threshold:.6f}")
    log_line(txt_path, f"n_top_anomaly_days: {int(out['is_top_anomaly'].sum())}")
    log_line(txt_path, f"n_devices: {df['kkt_id'].nunique()}")
    log_line(txt_path, f"Saved ranked KKT-days: {csv_path}")
    log_line(txt_path, f"Saved top KKT-days: {top_csv_path}")
    log_line(txt_path, f"Saved device scores: {device_csv_path}")
    log_line(txt_path, f"Runtime sec: {runtime:.2f}")

    return result


# For first full run keep max_rows=None.
# If memory is tight, set MAX_TS_ROWS = 500_000 for debugging.
MAX_TS_ROWS = None

ts_results = []

for dataset_name, meta in TS_DATASETS.items():
    path = meta["path"]
    if not path.exists():
        print("[MISSING]", path)
        continue

    print("\n" + "=" * 120)
    print(f"BUILD TS FEATURES: {dataset_name}")
    print("=" * 120)

    df_ts = build_ts_features(dataset_name, path, max_rows=MAX_TS_ROWS, force=FORCE)

    print("TS shape:", df_ts.shape)
    print("Unique kkt:", df_ts["kkt_id"].nunique())
    print("Date range:", df_ts["date"].min(), "->", df_ts["date"].max())

    for group in TS_FEATURE_GROUPS:
        for contamination in CONTAMINATION_LEVELS:
            print("\n" + "=" * 120)
            print(f"TS RUN dataset={dataset_name}, group={group}, contamination={contamination}")
            print("=" * 120)

            res = run_ts_anomaly_experiment(
                dataset_name=dataset_name,
                df=df_ts,
                group=group,
                contamination=contamination,
                force=FORCE,
            )
            ts_results.append(res)


BUILD TS FEATURES: ts_kkt_47_11
[2026-05-17 19:49:38] [SKIP] Loading cached csv: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\cache\ts_kkt_47_11__ts_features.csv
TS shape: (3168756, 93)
Unique kkt: 34443
Date range: 2023-10-01 00:00:00 -> 2023-12-31 00:00:00

TS RUN dataset=ts_kkt_47_11, group=all_numeric, contamination=0.01
[2026-05-17 19:51:06] [SKIP] existing run: ts_kkt_47_11__all_numeric__contam_0.01

TS RUN dataset=ts_kkt_47_11, group=all_numeric, contamination=0.03
[2026-05-17 19:51:06] [SKIP] existing run: ts_kkt_47_11__all_numeric__contam_0.03

TS RUN dataset=ts_kkt_47_11, group=all_numeric, contamination=0.05
[2026-05-17 19:51:06] [SKIP] existing run: ts_kkt_47_11__all_numeric__contam_0.05

TS RUN dataset=ts_kkt_47_11, group=core_daily, contamination=0.01
[2026-05-17 19:51:06] ========================================================================================================================
[2026-05-17 19:51:06] RUN TS dataset=ts_kkt_47_11, group=core_

In [20]:
# ============================================================
# Collect all results
# ============================================================

def collect_json_results() -> pd.DataFrame:
    rows = []

    for p in sorted(JSON_DIR.glob("*.json")):
        try:
            r = load_json(p)
        except Exception as e:
            rows.append({
                "json_file": str(p),
                "status": "bad_json",
                "error": repr(e),
            })
            continue

        row = {
            "json_file": str(p),
            "status": r.get("status"),
            "stage": r.get("stage"),
            "dataset": r.get("dataset"),
            "group": r.get("group"),
            "contamination": r.get("contamination"),
            "n_rows": r.get("n_rows"),
            "n_devices": r.get("n_devices"),
            "n_features": r.get("n_features"),
            "threshold": r.get("threshold"),
            "n_top_anomalies": r.get("n_top_anomalies"),
            "n_top_anomaly_days": r.get("n_top_anomaly_days"),
            "top_score": r.get("top_score"),
            "median_score": r.get("median_score"),
            "runtime_sec": r.get("runtime_sec"),
            "ranked_csv": r.get("ranked_csv"),
            "top_csv": r.get("top_csv"),
            "device_csv": r.get("device_csv"),
            "error": r.get("error"),
        }

        rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["stage", "dataset", "group", "contamination"], na_position="last").reset_index(drop=True)
    return df


results_df = collect_json_results()

save_df_csv(results_df, SUMMARY_DIR / "all_kkt_anomaly_runs_summary.csv")

try:
    results_df.to_excel(SUMMARY_DIR / "all_kkt_anomaly_runs_summary.xlsx", index=False)
    print("[SAVED]", SUMMARY_DIR / "all_kkt_anomaly_runs_summary.xlsx")
except Exception as e:
    print("[WARN] Excel save failed:", repr(e))

display(results_df)

[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\all_kkt_anomaly_runs_summary.csv (108, 19)
[WARN] Excel save failed: ModuleNotFoundError("No module named 'openpyxl'")


,json_file,status,stage,dataset,group,contamination,n_rows,n_devices,n_features,threshold,n_top_anomalies,n_top_anomaly_days,top_score,median_score,runtime_sec,ranked_csv,top_csv,device_csv,error
0,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,agg_kkt_level,agg_kkt_47_11,all_numeric,0.01,34443,NaN,150,0.999888,345.0,NaN,1.0,0.159773,21.924367,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,None
1,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,agg_kkt_level,agg_kkt_47_11,all_numeric,0.03,34443,NaN,150,0.648953,1034.0,NaN,1.0,0.159773,18.555602,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,None
2,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,agg_kkt_level,agg_kkt_47_11,all_numeric,0.05,34443,NaN,150,0.514184,1723.0,NaN,1.0,0.159773,19.884701,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,None
3,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,agg_kkt_level,agg_kkt_47_11,bills,0.01,34443,NaN,11,0.999953,345.0,NaN,1.0,0.158344,24.627632,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,None
4,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,agg_kkt_level,agg_kkt_47_11,bills,0.03,34443,NaN,11,0.885933,1034.0,NaN,1.0,0.158344,26.645549,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,agg_kkt_47_19,shape_complexity,0.03,15491,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
104,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,agg_kkt_47_19,shape_complexity,0.05,15491,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
105,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,agg_kkt_47_19,temporal,0.01,15491,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
106,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,agg_kkt_47_19,temporal,0.03,15491,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None


In [21]:
# ============================================================
# Diagnostics
# ============================================================

results_df = collect_json_results()

print("=" * 120)
print("STATUS COUNTS")
print("=" * 120)
display(results_df.groupby(["stage", "dataset", "status"], dropna=False).size().reset_index(name="count"))

print("=" * 120)
print("ERRORS")
print("=" * 120)
err = results_df[results_df["status"] != "ok"]
if len(err) == 0:
    print("No errors.")
else:
    display(err[["stage", "dataset", "group", "contamination", "status", "error", "json_file"]])

STATUS COUNTS


,stage,dataset,status,count
0,agg_kkt_level,agg_kkt_47_11,ok,21
1,agg_kkt_level,agg_kkt_47_19,ok,21
2,ts_kkt_day_level,ts_kkt_47_11,ok,21
3,ts_kkt_day_level,ts_kkt_47_19,ok,21
4,NaN,agg_kkt_47_11,skipped_no_features,12
5,NaN,agg_kkt_47_19,skipped_no_features,12


ERRORS


,stage,dataset,group,contamination,status,error,json_file
84,None,agg_kkt_47_11,benford,0.01,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
85,None,agg_kkt_47_11,benford,0.03,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
86,None,agg_kkt_47_11,benford,0.05,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
87,None,agg_kkt_47_11,cross_corr,0.01,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
88,None,agg_kkt_47_11,cross_corr,0.03,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
89,None,agg_kkt_47_11,cross_corr,0.05,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
90,None,agg_kkt_47_11,shape_complexity,0.01,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
91,None,agg_kkt_47_11,shape_complexity,0.03,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
92,None,agg_kkt_47_11,shape_complexity,0.05,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
93,None,agg_kkt_47_11,temporal,0.01,skipped_no_features,None,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...


In [22]:
# ============================================================
# Merge top aggregated KKT anomalies across runs
# ============================================================

def read_top_files(stage_prefix: str, max_per_file: int = 100) -> pd.DataFrame:
    dfs = []

    for p in sorted(CSV_DIR.glob(f"{stage_prefix}__*__top*.csv")):
        try:
            df = pd.read_csv(p)
            df["_source_top_file"] = p.name
            dfs.append(df.head(max_per_file))
        except Exception as e:
            print("[WARN] failed reading", p, repr(e))

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


agg_top_all = read_top_files("agg", max_per_file=200)

if not agg_top_all.empty:
    save_df_csv(agg_top_all, SUMMARY_DIR / "all_top_agg_kkt_anomalies_raw.csv")

    # Consensus across runs by KKT.
    score_cols = [c for c in agg_top_all.columns if c.startswith("score_")] + ["anomaly_score"]
    score_cols = list(dict.fromkeys([c for c in score_cols if c in agg_top_all.columns]))

    group_cols = ["kkt_id"]
    for c in ["trade_store_hash", "trade_object_hash", "okved2", "region", "type", "category"]:
        if c in agg_top_all.columns:
            group_cols.append(c)

    agg_consensus = (
        agg_top_all
        .groupby(group_cols, dropna=False)
        .agg(
            n_appearances=("_source_top_file", "nunique"),
            best_score=("anomaly_score", "max"),
            mean_score=("anomaly_score", "mean"),
            best_rank=("anomaly_rank", "min") if "anomaly_rank" in agg_top_all.columns else ("anomaly_score", "max"),
            explanation=("top_feature_explanation", lambda s: " || ".join(pd.Series(s).dropna().astype(str).head(3))),
            source_files=("_source_top_file", lambda s: " | ".join(sorted(set(map(str, s)))[:10])),
        )
        .reset_index()
    )

    agg_consensus["final_kkt_anomaly_score"] = normalize_score(
        0.50 * normalize_score(agg_consensus["best_score"].to_numpy())
        + 0.30 * normalize_score(agg_consensus["mean_score"].to_numpy())
        + 0.20 * normalize_score(agg_consensus["n_appearances"].to_numpy())
    )

    agg_consensus = agg_consensus.sort_values("final_kkt_anomaly_score", ascending=False).reset_index(drop=True)
    agg_consensus["final_rank"] = np.arange(1, len(agg_consensus) + 1)

    save_df_csv(agg_consensus, SUMMARY_DIR / "final_agg_kkt_anomaly_ranking.csv")

    display(agg_consensus.head(100))
else:
    print("No aggregated top anomaly files found yet.")

[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\all_top_agg_kkt_anomalies_raw.csv (8078, 172)
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\final_agg_kkt_anomaly_ranking.csv (1248, 15)


,kkt_id,trade_store_hash,trade_object_hash,okved2,region,type,category,n_appearances,best_score,mean_score,best_rank,explanation,source_files,final_kkt_anomaly_score,final_rank
0,3cee44bca7154720a66702cce1c365fb,x0248cae35b0f1b13cb666f4a219984cd7257096ab26b5...,x2e536953573e74678da072d67710c7ced363838f0590a...,47.19,77,1,3,18,1.0,1.0,34,std_sum__standard_deviation=299.87; std_sum__m...,agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,1.000000,1
1,a9943e26a702150fa9b590be7b95fe00,x07ee8e2dd5ea21631d26b0940f1d7af41275e1b0630ac...,x8e15c10874d46830f09a6059b4b5b0082b2d48900518e...,47.11,77,1,1,18,1.0,1.0,220,std_sum__standard_deviation=776.89; std_sum__a...,agg__agg_kkt_47_11_all_numeric_contam_0.01__to...,1.000000,2
2,58ec45cf3fe5aef3ce9dec80e1d1af31,x480c04ba9bc7af8cc79395648168a059414d4420db227...,xbdb7474685fdb7c778b9b847ed6e1c4d86e04b56e39ee...,47.19,77,2,1,18,1.0,1.0,56,min_sum__maximum=636.88; min_sum__standard_dev...,agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,1.000000,3
3,c3209ded41482c3a07e988ba56be748e,x670411494f8c56bc027c78c0deda9ff298d995d6d42fd...,x729b873c3c742aa5fe72235287f8ed29bb06236c39d74...,47.19,77,2,1,18,1.0,1.0,112,"min_sum__fft_coefficient__attr_""abs""__coeff_1=...",agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,1.000000,4
4,58af36c66282c092268aff6d4417a8af,x91455afc66c1600f405373423adb257c0dc6d9f350b58...,x226e5dec6a1380beef0bfa392214b8129362180f40996...,47.19,77,1,2,18,1.0,1.0,55,min_sum__maximum=499.93; min_sum__standard_dev...,agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,1.000000,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,c4d491dfab29cd2975ea3d3e8ab4e8eb,c4d491dfab29cd2975ea3d3e8ab4e8eb,c4d491dfab29cd2975ea3d3e8ab4e8eb,47.19,77,1,1,15,1.0,1.0,113,min_sum__maximum=777.47; median_sum__agg_linea...,agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,0.960526,96
96,5da86218ffedb2c8de6bbcc8f5eff8a9,x5bf4538e6e2f9936c923e6021e2e0762268aa7f3131d8...,x39a3c8ab7d9e1e4a49bbd0ae356f9cd14c38a4e9a80a8...,47.11,77,2,1,15,1.0,1.0,127,min_sum__minimum=4700.00; avg_price_piece__mea...,agg__agg_kkt_47_11_all_numeric_contam_0.01__to...,0.960526,97
97,c1184bf45b33679dc6b3e5446b75d20f,xfe32e366c4ce19bcf0e2efbdfd5aa3269cf4bcbe9f57f...,x6aaa3c380ab2de1706ac1d4bd438c8d842b986e8a3a28...,47.11,77,2,1,15,1.0,1.0,260,avg_price_item__standard_deviation=3127.03; st...,agg__agg_kkt_47_11_all_numeric_contam_0.01__to...,0.960526,98
98,2aef0d75d12248c2dd12dd458c95a366,x89ab1748efd5488fcbe94f14980f3cfa95d03be5c46b4...,x84cebcd98ae038a0b04d689c953ecfce3e4b0cd635def...,47.19,77,1,2,15,1.0,1.0,21,std_sum__mean=79.69; std_sum__median=77.31; st...,agg__agg_kkt_47_19_all_numeric_contam_0.01__to...,0.960526,99


In [23]:
# ============================================================
# Merge top KKT-day anomalies across time-series runs
# ============================================================

ts_top_all = read_top_files("ts", max_per_file=500)

if not ts_top_all.empty:
    save_df_csv(ts_top_all, SUMMARY_DIR / "all_top_ts_kkt_day_anomalies_raw.csv")

    key_cols = ["kkt_id", "date"]
    for c in ["trade_store_hash", "trade_object_hash"]:
        if c in ts_top_all.columns:
            key_cols.append(c)

    ts_consensus = (
        ts_top_all
        .groupby(key_cols, dropna=False)
        .agg(
            n_appearances=("_source_top_file", "nunique"),
            best_day_score=("anomaly_score", "max"),
            mean_day_score=("anomaly_score", "mean"),
            total_sum=("total_sum", "max") if "total_sum" in ts_top_all.columns else ("anomaly_score", "size"),
            n_bills=("n_bills", "max") if "n_bills" in ts_top_all.columns else ("anomaly_score", "size"),
            total_ecash=("total_ecash", "max") if "total_ecash" in ts_top_all.columns else ("anomaly_score", "size"),
            ecash_fraction=("ecash_fraction", "max") if "ecash_fraction" in ts_top_all.columns else ("anomaly_score", "size"),
            explanation=("top_feature_explanation", lambda s: " || ".join(pd.Series(s).dropna().astype(str).head(3))),
            source_files=("_source_top_file", lambda s: " | ".join(sorted(set(map(str, s)))[:10])),
        )
        .reset_index()
    )

    ts_consensus["final_day_anomaly_score"] = normalize_score(
        0.50 * normalize_score(ts_consensus["best_day_score"].to_numpy())
        + 0.30 * normalize_score(ts_consensus["mean_day_score"].to_numpy())
        + 0.20 * normalize_score(ts_consensus["n_appearances"].to_numpy())
    )

    ts_consensus = ts_consensus.sort_values("final_day_anomaly_score", ascending=False).reset_index(drop=True)
    ts_consensus["final_day_rank"] = np.arange(1, len(ts_consensus) + 1)

    save_df_csv(ts_consensus, SUMMARY_DIR / "final_ts_kkt_day_anomaly_ranking.csv")

    display(ts_consensus.head(200))
else:
    print("No time-series top anomaly files found yet.")

[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\all_top_ts_kkt_day_anomalies_raw.csv (21000, 34)
[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\final_ts_kkt_day_anomaly_ranking.csv (6791, 15)


,kkt_id,date,trade_store_hash,trade_object_hash,n_appearances,best_day_score,mean_day_score,total_sum,n_bills,total_ecash,ecash_fraction,explanation,source_files,final_day_anomaly_score,final_day_rank
0,a32f067711de2e6a4eab509c3590f090,2023-12-31,x72df01080c58dc3aa91e6cd9c9400aa64badb12bfa9f9...,x174c4cadcad18eb2e89033ac22babb590f39533dce3df...,6,1.0,1.0,1287178.4,413.0,1081986.4,0.840588,total_cash=14.30; total_sum=12.38; total_ecash...,ts__ts_kkt_47_11_core_daily_contam_0.01__top_k...,1.0,1
1,dc978bcbb6686f0e70c173d2367e131a,2023-12-26,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,94620.0,1.0,0.0,0.000000,min_sum=1819.45; avg_price_piece=436.98; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,2
2,dc978bcbb6686f0e70c173d2367e131a,2023-12-18,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,101982.0,2.0,0.0,0.000000,avg_price_piece=235.29; min_sum=197.95; avg_pr...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,3
3,dc978bcbb6686f0e70c173d2367e131a,2023-12-19,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,92560.0,1.0,0.0,0.000000,min_sum=1779.84; avg_price_piece=427.45; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,4
4,45b5abf828e4410557284e89ab1223b8,2023-12-28,xfe3c5105cd73db6bf3265e3c81af777c08cfae0d735e2...,xf73ece543d2d107e1f8d56f1d34a28c687d455aa1066d...,6,1.0,1.0,94470.0,4.0,94470.0,1.000000,total_sum__roll7_rel_jump=240227123200.00; tot...,ts__ts_kkt_47_19_all_numeric_contam_0.01__top_...,1.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,af28838b75778227b973f7a978e934fa,2023-11-02,af28838b75778227b973f7a978e934fa,af28838b75778227b973f7a978e934fa,3,1.0,1.0,1699000.0,17.0,0.0,0.000000,min_sum=1711.38; avg_price_piece=461.58; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,0.0,196
196,a8b4cd4a3838e87fb025359cee378909,2023-12-01,x2ec6f4806d7442ff77b456a5866aae1211ecc1acb5b2c...,x46c36ade1e18d566f67478d4aa89290d47f6070c1d14b...,3,1.0,1.0,83835.0,41.0,0.0,0.000000,derived_cash_to_ecash=255164594257920.00; tota...,ts__ts_kkt_47_11_all_numeric_contam_0.01__top_...,0.0,197
197,a8b4cd4a3838e87fb025359cee378909,2023-11-30,x2ec6f4806d7442ff77b456a5866aae1211ecc1acb5b2c...,x46c36ade1e18d566f67478d4aa89290d47f6070c1d14b...,3,1.0,1.0,100002.0,36.0,0.0,0.000000,derived_cash_to_ecash=304371346702336.00; min_...,ts__ts_kkt_47_11_all_numeric_contam_0.01__top_...,0.0,198
198,a8b4cd4a3838e87fb025359cee378909,2023-11-29,x2ec6f4806d7442ff77b456a5866aae1211ecc1acb5b2c...,x46c36ade1e18d566f67478d4aa89290d47f6070c1d14b...,3,1.0,1.0,66255.0,33.0,0.0,0.000000,derived_cash_to_ecash=201657187041280.00; min_...,ts__ts_kkt_47_11_all_numeric_contam_0.01__top_...,0.0,199


In [24]:
# ============================================================
# Combine aggregate KKT ranking and time-series device ranking
# ============================================================

# Read all device-level scores from ts runs.
device_files = sorted(CSV_DIR.glob("ts__*__device_aggregated_scores.csv"))

device_dfs = []

for p in device_files:
    try:
        df = pd.read_csv(p)
        df["_source_device_file"] = p.name
        device_dfs.append(df)
    except Exception as e:
        print("[WARN] failed reading", p, repr(e))

if device_dfs:
    ts_device_all = pd.concat(device_dfs, ignore_index=True)

    ts_device_consensus = (
        ts_device_all
        .groupby("kkt_id", dropna=False)
        .agg(
            ts_n_appearances=("_source_device_file", "nunique"),
            ts_best_device_score=("device_score_combined", "max"),
            ts_mean_device_score=("device_score_combined", "mean"),
            ts_max_daily_score=("device_score_max", "max"),
            ts_total_anomaly_days=("n_top_anomaly_days", "sum"),
            ts_total_days=("n_days", "max"),
            ts_source_files=("_source_device_file", lambda s: " | ".join(sorted(set(map(str, s)))[:10])),
        )
        .reset_index()
    )

    ts_device_consensus["ts_final_score"] = normalize_score(
        0.45 * normalize_score(ts_device_consensus["ts_best_device_score"].to_numpy())
        + 0.25 * normalize_score(ts_device_consensus["ts_mean_device_score"].to_numpy())
        + 0.20 * normalize_score(ts_device_consensus["ts_total_anomaly_days"].to_numpy())
        + 0.10 * normalize_score(ts_device_consensus["ts_n_appearances"].to_numpy())
    )

    save_df_csv(ts_device_consensus, SUMMARY_DIR / "final_ts_device_anomaly_ranking.csv")
    display(ts_device_consensus.sort_values("ts_final_score", ascending=False).head(100))

else:
    ts_device_consensus = pd.DataFrame()
    print("No ts device files found.")


# Combine with aggregate profile anomalies.
agg_path = SUMMARY_DIR / "final_agg_kkt_anomaly_ranking.csv"

if agg_path.exists():
    agg_consensus = pd.read_csv(agg_path)
else:
    agg_consensus = pd.DataFrame()

if not agg_consensus.empty and not ts_device_consensus.empty:
    combined = pd.merge(
        agg_consensus,
        ts_device_consensus,
        on="kkt_id",
        how="outer",
        suffixes=("_agg", "_ts"),
    )

    combined["final_kkt_anomaly_score"] = normalize_score(
        0.50 * normalize_score(combined["final_kkt_anomaly_score"].fillna(0).to_numpy())
        + 0.50 * normalize_score(combined["ts_final_score"].fillna(0).to_numpy())
    )

    combined = combined.sort_values("final_kkt_anomaly_score", ascending=False).reset_index(drop=True)
    combined["final_rank"] = np.arange(1, len(combined) + 1)

    save_df_csv(combined, SUMMARY_DIR / "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv")
    display(combined.head(200))

elif not agg_consensus.empty:
    print("Only aggregate consensus exists.")
    display(agg_consensus.head(100))

elif not ts_device_consensus.empty:
    print("Only time-series device consensus exists.")
    display(ts_device_consensus.sort_values("ts_final_score", ascending=False).head(100))

else:
    print("No combined results yet.")

[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\final_ts_device_anomaly_ranking.csv (49934, 9)


,kkt_id,ts_n_appearances,ts_best_device_score,ts_mean_device_score,ts_max_daily_score,ts_total_anomaly_days,ts_total_days,ts_source_files,ts_final_score
24967,800065ec054bba34d4ef073e09e77af8,21,1.0,0.783455,1.0,1264,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
41570,d526fa56440d20a31d722cfd2a29d96e,21,1.0,0.921461,1.0,1194,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
46348,edaa203e45d35c46f8caeeb8f5dafdf5,21,1.0,0.815754,1.0,1071,92,ts__ts_kkt_47_19_all_numeric_contam_0.01__devi...,1.0
46340,eda52b6b8a1df4761050ae10a1c2d75c,21,1.0,0.906797,1.0,679,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
46326,ed910dc58be23f111117a2a51b1dc4db,21,1.0,0.779617,1.0,1290,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
...,...,...,...,...,...,...,...,...,...
21265,6cef148d99d96252953464563f92bbd3,21,1.0,0.802274,1.0,1420,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
34867,b2a0ec2c332d6fc90f151dd875b4deca,21,1.0,0.899518,1.0,766,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.0
5210,1abfac7cf8286e70bbab1a3fc97bc305,21,1.0,0.769518,1.0,1258,92,ts__ts_kkt_47_19_all_numeric_contam_0.01__devi...,1.0
31458,a0f5efba84b14e5d379e975e0480dc50,21,1.0,0.879409,1.0,1346,92,ts__ts_kkt_47_19_all_numeric_contam_0.01__devi...,1.0


[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_COMBINED_KKT_ANOMALY_RANKING.csv (49934, 23)


,kkt_id,trade_store_hash,trade_object_hash,okved2,region,type,category,n_appearances,best_score,mean_score,...,final_kkt_anomaly_score,final_rank,ts_n_appearances,ts_best_device_score,ts_mean_device_score,ts_max_daily_score,ts_total_anomaly_days,ts_total_days,ts_source_files,ts_final_score
0,800065ec054bba34d4ef073e09e77af8,xc176f354cfc63fd15648e5452efaec15cd92092d18ace...,xf45b3fe48137b26547ffbb1101a24c1a511a759dc8667...,47.11,77.0,1.0,4.0,15.0,1.0,1.000000,...,1.0,1,21,1.0,0.783455,1.0,1264,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
1,69ca5dcd89e9ef68ec5142a3529fabaf,xe22c63cfea722ab62ce8d73a1a312038f5f764dbe0136...,xc9615574a9427a10ad3aa8289f944e1134f974798fd07...,47.11,77.0,2.0,1.0,3.0,1.0,1.000000,...,1.0,2,21,1.0,0.926616,1.0,779,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
2,99399be9dc9b613d22e98abd058f1b0a,99399be9dc9b613d22e98abd058f1b0a,99399be9dc9b613d22e98abd058f1b0a,47.11,77.0,2.0,1.0,6.0,1.0,1.000000,...,1.0,3,21,1.0,0.925520,1.0,1145,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
3,4eeb3e43020e0bc7b8f9041b143ce9a3,x4a5fce54fc8dbb04049a7bd7fc45e42bd2db295b1c150...,x7f13dc11504913ad3f9017e21a36887709c7f12605fcb...,47.11,77.0,1.0,4.0,9.0,1.0,1.000000,...,1.0,4,21,1.0,0.601524,1.0,802,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,0.947564
4,c55bd5d73bee168680fe8cfa36ea7bd1,x34d43310a2fef19bc178bd15d3299cda0ec278a777a99...,xf7fbbd70758e4cdbbadee25c24171c73c407614395716...,47.11,77.0,2.0,1.0,3.0,1.0,1.000000,...,1.0,5,21,1.0,0.801229,1.0,776,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,4bfca0565ec0fbe35d48f5bb568ffefe,x28939c55fd3b65cd72a8b811aba4f2156a7cf2b1356aa...,xd357aea1ac00a95f876cfe102b83bb921820fa7572d05...,47.19,77.0,2.0,1.0,3.0,1.0,1.000000,...,1.0,196,21,1.0,0.625209,1.0,767,92,ts__ts_kkt_47_19_all_numeric_contam_0.01__devi...,0.945451
196,0c7f3fd1f2214facee9c4acc0369f6da,x652bb8b82e6c5b3b6cb4e1bfd366518a369bb62411341...,x87be24d6d535858134ad68cb9492ba4d54adf22bd4afa...,47.11,77.0,1.0,1.0,3.0,1.0,1.000000,...,1.0,197,21,1.0,0.881526,1.0,893,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
197,4bc98e1df6e212a41c4b67ca93c14ea4,xb7ee8c6cd12e96dabe2ad34d7da3c9545095106660538...,x4b6f06ae9c7116fb7cf4a1c146557dcb26320c1713558...,47.11,77.0,1.0,4.0,3.0,1.0,1.000000,...,1.0,198,21,1.0,0.758428,1.0,1154,92,ts__ts_kkt_47_11_all_numeric_contam_0.01__devi...,1.000000
198,4c8bd523897010f03aa4e93808390c47,4c8bd523897010f03aa4e93808390c47,4c8bd523897010f03aa4e93808390c47,47.19,77.0,2.0,1.0,6.0,1.0,1.000000,...,1.0,199,21,1.0,0.816674,1.0,629,92,ts__ts_kkt_47_19_all_numeric_contam_0.01__devi...,0.969895


In [25]:
# ============================================================
# Ablation-style summary: which feature groups produce strongest anomalies
# ============================================================

results_df = collect_json_results()
ok = results_df[results_df["status"] == "ok"].copy()

# Since unsupervised has no labels, compare:
# - number of top anomalies
# - top score
# - median score
# - runtime
# - overlap of top-K anomalies later
display(ok[[
    "stage",
    "dataset",
    "group",
    "contamination",
    "n_rows",
    "n_devices",
    "n_features",
    "n_top_anomalies",
    "n_top_anomaly_days",
    "top_score",
    "median_score",
    "runtime_sec",
]])

save_df_csv(ok, SUMMARY_DIR / "ablation_feature_group_run_summary.csv")

,stage,dataset,group,contamination,n_rows,n_devices,n_features,n_top_anomalies,n_top_anomaly_days,top_score,median_score,runtime_sec
0,agg_kkt_level,agg_kkt_47_11,all_numeric,0.01,34443,NaN,150,345.0,NaN,1.0,0.159773,21.924367
1,agg_kkt_level,agg_kkt_47_11,all_numeric,0.03,34443,NaN,150,1034.0,NaN,1.0,0.159773,18.555602
2,agg_kkt_level,agg_kkt_47_11,all_numeric,0.05,34443,NaN,150,1723.0,NaN,1.0,0.159773,19.884701
3,agg_kkt_level,agg_kkt_47_11,bills,0.01,34443,NaN,11,345.0,NaN,1.0,0.158344,24.627632
4,agg_kkt_level,agg_kkt_47_11,bills,0.03,34443,NaN,11,1034.0,NaN,1.0,0.158344,26.645549
...,...,...,...,...,...,...,...,...,...,...,...,...
79,ts_kkt_day_level,ts_kkt_47_19,rolling_deviation,0.03,1425172,15491.0,48,NaN,42756.0,1.0,0.054617,139.804998
80,ts_kkt_day_level,ts_kkt_47_19,rolling_deviation,0.05,1425172,15491.0,48,NaN,71259.0,1.0,0.054617,138.551002
81,ts_kkt_day_level,ts_kkt_47_19,turnover,0.01,1425172,15491.0,42,NaN,14252.0,1.0,0.209810,144.008996
82,ts_kkt_day_level,ts_kkt_47_19,turnover,0.03,1425172,15491.0,42,NaN,42756.0,1.0,0.209810,141.808001


[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\ablation_feature_group_run_summary.csv (84, 19)


In [26]:
# ============================================================
# Top-K overlap analysis between ablations
# ============================================================

def top_ids_from_csv(path: str, id_col: str = "kkt_id", top_k: int = 200) -> set:
    df = pd.read_csv(path)
    if id_col not in df.columns:
        return set()
    return set(df[id_col].head(top_k).astype(str))


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))


overlap_rows = []

ok = collect_json_results()
ok = ok[ok["status"] == "ok"].copy()

for stage in ["agg_kkt_level", "ts_kkt_day_level"]:
    stage_df = ok[ok["stage"] == stage].copy()

    for dataset in sorted(stage_df["dataset"].dropna().unique()):
        for contamination in sorted(stage_df["contamination"].dropna().unique()):
            sub = stage_df[
                (stage_df["dataset"] == dataset)
                & (stage_df["contamination"] == contamination)
            ]

            # Compare all_numeric with other groups.
            base = sub[sub["group"] == "all_numeric"]

            if base.empty:
                continue

            base_path = base.iloc[0]["top_csv"]
            if not isinstance(base_path, str) or not Path(base_path).exists():
                continue

            base_ids = top_ids_from_csv(base_path, id_col="kkt_id", top_k=200)

            for _, r in sub.iterrows():
                group = r["group"]
                path = r["top_csv"]

                if not isinstance(path, str) or not Path(path).exists():
                    continue

                ids = top_ids_from_csv(path, id_col="kkt_id", top_k=200)

                overlap_rows.append({
                    "stage": stage,
                    "dataset": dataset,
                    "contamination": contamination,
                    "base_group": "all_numeric",
                    "group": group,
                    "top_k": 200,
                    "jaccard_with_all_numeric": jaccard(base_ids, ids),
                    "intersection": len(base_ids & ids),
                    "base_size": len(base_ids),
                    "group_size": len(ids),
                    "top_csv": path,
                })

overlap_df = pd.DataFrame(overlap_rows)
save_df_csv(overlap_df, SUMMARY_DIR / "topk_overlap_with_all_numeric.csv")
display(overlap_df.sort_values(["stage", "dataset", "contamination", "jaccard_with_all_numeric"], ascending=[True, True, True, False]))

[SAVED] C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\topk_overlap_with_all_numeric.csv (84, 11)


,stage,dataset,contamination,base_group,group,top_k,jaccard_with_all_numeric,intersection,base_size,group_size,top_csv
0,agg_kkt_level,agg_kkt_47_11,0.01,all_numeric,all_numeric,200,1.000000,200,200,200,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
3,agg_kkt_level,agg_kkt_47_11,0.01,all_numeric,non_temporal,200,1.000000,200,200,200,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
5,agg_kkt_level,agg_kkt_47_11,0.01,all_numeric,simple_core,200,0.444043,123,200,200,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
6,agg_kkt_level,agg_kkt_47_11,0.01,all_numeric,turnover,200,0.342282,102,200,200,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
2,agg_kkt_level,agg_kkt_47_11,0.01,all_numeric,items_price,200,0.183432,62,200,200,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
...,...,...,...,...,...,...,...,...,...,...,...
79,ts_kkt_day_level,ts_kkt_47_19,0.05,all_numeric,core_daily,200,0.009259,1,82,27,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
81,ts_kkt_day_level,ts_kkt_47_19,0.05,all_numeric,peer_deviation,200,0.009091,1,82,29,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
82,ts_kkt_day_level,ts_kkt_47_19,0.05,all_numeric,rolling_deviation,200,0.005682,1,82,95,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
83,ts_kkt_day_level,ts_kkt_47_19,0.05,all_numeric,turnover,200,0.005682,1,82,95,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...


In [27]:
# ============================================================
# Final TXT report
# ============================================================

report_path = SUMMARY_DIR / "KKT_ANOMALY_DETECTION_REPORT.txt"

results_df = collect_json_results()

with open(report_path, "w", encoding="utf-8") as f:
    f.write("KKT Anomaly Detection Report\n")
    f.write("=" * 120 + "\n\n")

    f.write(f"Created: {now_str()}\n")
    f.write(f"Data root: {DATA_ROOT}\n")
    f.write(f"Output root: {OUT_ROOT}\n\n")

    f.write("Selected datasets:\n")
    for k, p in PATHS.items():
        if p.exists():
            f.write(f"- {k}: {p} ({file_size_mb(p):.2f} MB)\n")
    f.write("\n")

    f.write("Pipeline summary:\n")
    f.write("- Aggregated KKT-level anomaly detection using precomputed KKT descriptors.\n")
    f.write("- KKT-day anomaly detection using daily time-series, rolling deviations, and peer-date robust deviations.\n")
    f.write("- Feature-group ablations: turnover, bills/items, payment mix, temporal, Benford, shape/complexity, simple core.\n")
    f.write("- Models: IsolationForest, PCA reconstruction, robust-z distance, LOF when feasible, EllipticEnvelope when feasible.\n")
    f.write("- Final scores are consensus scores across several unsupervised detectors.\n\n")

    f.write("Run summary:\n")
    f.write("-" * 120 + "\n")
    if not results_df.empty:
        f.write(results_df[[
            "stage", "dataset", "group", "contamination", "status",
            "n_rows", "n_devices", "n_features",
            "n_top_anomalies", "n_top_anomaly_days",
            "top_score", "median_score", "runtime_sec"
        ]].to_string(index=False))
    f.write("\n\n")

    final_combined = SUMMARY_DIR / "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv"
    if final_combined.exists():
        f.write("Top combined KKT anomalies:\n")
        f.write("-" * 120 + "\n")
        top = pd.read_csv(final_combined).head(50)
        cols = [c for c in [
            "final_rank", "kkt_id", "final_kkt_anomaly_score",
            "best_score", "mean_score", "ts_final_score",
            "ts_total_anomaly_days", "explanation"
        ] if c in top.columns]
        f.write(top[cols].to_string(index=False))
        f.write("\n\n")

    top_days = SUMMARY_DIR / "final_ts_kkt_day_anomaly_ranking.csv"
    if top_days.exists():
        f.write("Top KKT-day anomalies:\n")
        f.write("-" * 120 + "\n")
        top = pd.read_csv(top_days).head(50)
        cols = [c for c in [
            "final_day_rank", "kkt_id", "date", "final_day_anomaly_score",
            "total_sum", "n_bills", "ecash_fraction", "explanation"
        ] if c in top.columns]
        f.write(top[cols].to_string(index=False))
        f.write("\n\n")

    overlap_path = SUMMARY_DIR / "topk_overlap_with_all_numeric.csv"
    if overlap_path.exists():
        f.write("Top-K overlap with all_numeric feature group:\n")
        f.write("-" * 120 + "\n")
        ov = pd.read_csv(overlap_path)
        f.write(ov.to_string(index=False))
        f.write("\n\n")

print("Saved report:", report_path)
print(report_path.read_text(encoding="utf-8")[:7000])

Saved report: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\KKT_ANOMALY_DETECTION_REPORT.txt
KKT Anomaly Detection Report

Created: 2026-05-17 22:19:20
Data root: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2
Output root: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs

Selected datasets:
- ts_kkt_47_11: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.11_region_77_impute_True.csv (1024.72 MB)
- ts_kkt_47_19: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_time_series\ts_kkt_id_okved_47.19_region_77_impute_True.csv (451.60 MB)
- agg_kkt_47_11: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.11_region_77_impute_True.csv (78.62 MB)
- agg_kkt_47_19: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2\dataloader_aggregated\agg_kkt_okved_47.

In [28]:
# ============================================================
# FULL CHECK: what was computed / missing / failed
# for KKT anomaly pipeline
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
from datetime import datetime

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

OUT_ROOT = Path.cwd() / "kkt_anomaly_outputs"

LOG_DIR = OUT_ROOT / "logs"
JSON_DIR = OUT_ROOT / "results_json"
CSV_DIR = OUT_ROOT / "results_csv"
SUMMARY_DIR = OUT_ROOT / "summary"
CACHE_DIR = OUT_ROOT / "cache"

# Expected experiment design from the previous pipeline
AGG_DATASETS = ["agg_kkt_47_11", "agg_kkt_47_19"]
TS_DATASETS = ["ts_kkt_47_11", "ts_kkt_47_19"]

AGG_FEATURE_GROUPS = [
    "all_numeric",
    "turnover",
    "bills",
    "items_price",
    "payment_mix",
    "temporal",
    "benford",
    "cross_corr",
    "shape_complexity",
    "non_temporal",
    "simple_core",
]

TS_FEATURE_GROUPS = [
    "all_numeric",
    "core_daily",
    "rolling_deviation",
    "peer_deviation",
    "payment_mix",
    "turnover",
    "bills_items",
]

CONTAMINATION_LEVELS = [0.01, 0.03, 0.05]

EXPECTED_AGG = len(AGG_DATASETS) * len(AGG_FEATURE_GROUPS) * len(CONTAMINATION_LEVELS)
EXPECTED_TS = len(TS_DATASETS) * len(TS_FEATURE_GROUPS) * len(CONTAMINATION_LEVELS)
EXPECTED_TOTAL = EXPECTED_AGG + EXPECTED_TS

print("=" * 140)
print("KKT ANOMALY PIPELINE COMPLETION CHECK")
print("=" * 140)
print("Time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("OUT_ROOT:", OUT_ROOT)
print("OUT_ROOT exists:", OUT_ROOT.exists())
print("Expected agg experiments:", EXPECTED_AGG)
print("Expected ts experiments:", EXPECTED_TS)
print("Expected total experiments:", EXPECTED_TOTAL)

if not OUT_ROOT.exists():
    raise FileNotFoundError(f"OUT_ROOT does not exist: {OUT_ROOT}")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    if path.is_file():
        return path.stat().st_size / 1024 / 1024
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / 1024 / 1024


def count_files(path: Path) -> int:
    if not path.exists():
        return 0
    if path.is_file():
        return 1
    return sum(1 for p in path.rglob("*") if p.is_file())


def safe_json_load(path: Path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, repr(e)


def safe_read_csv_head(path: Path, nrows=3):
    try:
        return pd.read_csv(path, nrows=nrows), None
    except Exception as e:
        return None, repr(e)


def expected_json_name(prefix, dataset, group, contamination):
    # This reproduces the naming logic enough for checking expected files.
    def safe_name(x):
        x = str(x)
        bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
        for b in bad:
            x = x.replace(b, "_")
        while "__" in x:
            x = x.replace("__", "_")
        return x.strip("_")
    run_id = f"{dataset}__{group}__contam_{contamination}"
    return f"{prefix}__{safe_name(run_id)}.json"


# ------------------------------------------------------------
# Directory summary
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("DIRECTORY SUMMARY")
print("=" * 140)

dir_rows = []
for d in [LOG_DIR, JSON_DIR, CSV_DIR, SUMMARY_DIR, CACHE_DIR]:
    dir_rows.append({
        "folder": d.name,
        "exists": d.exists(),
        "files": count_files(d),
        "size_mb": round(size_mb(d), 2),
        "path": str(d),
    })

dir_df = pd.DataFrame(dir_rows)
display(dir_df)

# ------------------------------------------------------------
# JSON result inventory
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("JSON RESULT INVENTORY")
print("=" * 140)

json_rows = []

if JSON_DIR.exists():
    for p in sorted(JSON_DIR.glob("*.json")):
        obj, err = safe_json_load(p)

        if obj is None:
            json_rows.append({
                "file": p.name,
                "path": str(p),
                "status": "bad_json",
                "error": err,
                "stage": None,
                "dataset": None,
                "group": None,
                "contamination": None,
                "n_rows": None,
                "n_devices": None,
                "n_features": None,
                "runtime_sec": None,
                "ranked_csv": None,
                "top_csv": None,
                "device_csv": None,
            })
            continue

        json_rows.append({
            "file": p.name,
            "path": str(p),
            "status": obj.get("status"),
            "error": obj.get("error"),
            "stage": obj.get("stage"),
            "dataset": obj.get("dataset"),
            "group": obj.get("group"),
            "contamination": obj.get("contamination"),
            "n_rows": obj.get("n_rows"),
            "n_devices": obj.get("n_devices"),
            "n_features": obj.get("n_features"),
            "n_top_anomalies": obj.get("n_top_anomalies"),
            "n_top_anomaly_days": obj.get("n_top_anomaly_days"),
            "runtime_sec": obj.get("runtime_sec"),
            "ranked_csv": obj.get("ranked_csv"),
            "top_csv": obj.get("top_csv"),
            "device_csv": obj.get("device_csv"),
        })

json_df = pd.DataFrame(json_rows)

if json_df.empty:
    print("No JSON result files found.")
else:
    print("JSON files:", len(json_df))
    display(json_df.head(20))

    print("\nStatus counts:")
    display(
        json_df
        .groupby(["stage", "status"], dropna=False)
        .size()
        .reset_index(name="count")
    )

    print("\nDataset/group completion counts:")
    display(
        json_df
        .groupby(["stage", "dataset"], dropna=False)
        .size()
        .reset_index(name="n_json")
    )

# ------------------------------------------------------------
# Expected vs actual experiments
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("EXPECTED VS ACTUAL EXPERIMENTS")
print("=" * 140)

expected_rows = []

for dataset in AGG_DATASETS:
    for group in AGG_FEATURE_GROUPS:
        for contam in CONTAMINATION_LEVELS:
            expected_rows.append({
                "stage": "agg_kkt_level",
                "prefix": "agg",
                "dataset": dataset,
                "group": group,
                "contamination": contam,
                "expected_json": expected_json_name("agg", dataset, group, contam),
            })

for dataset in TS_DATASETS:
    for group in TS_FEATURE_GROUPS:
        for contam in CONTAMINATION_LEVELS:
            expected_rows.append({
                "stage": "ts_kkt_day_level",
                "prefix": "ts",
                "dataset": dataset,
                "group": group,
                "contamination": contam,
                "expected_json": expected_json_name("ts", dataset, group, contam),
            })

expected_df = pd.DataFrame(expected_rows)

actual_by_file = {}
if not json_df.empty:
    actual_by_file = {row["file"]: row for _, row in json_df.iterrows()}

check_rows = []

for _, r in expected_df.iterrows():
    fname = r["expected_json"]
    json_path = JSON_DIR / fname
    exists = json_path.exists()

    obj = None
    err = None
    if exists:
        obj, err = safe_json_load(json_path)

    status = None if obj is None else obj.get("status")
    ranked_csv = None if obj is None else obj.get("ranked_csv")
    top_csv = None if obj is None else obj.get("top_csv")
    device_csv = None if obj is None else obj.get("device_csv")

    ranked_exists = Path(ranked_csv).exists() if isinstance(ranked_csv, str) else False
    top_exists = Path(top_csv).exists() if isinstance(top_csv, str) else False
    device_exists = Path(device_csv).exists() if isinstance(device_csv, str) else False

    check_rows.append({
        "stage": r["stage"],
        "dataset": r["dataset"],
        "group": r["group"],
        "contamination": r["contamination"],
        "expected_json": fname,
        "json_exists": exists,
        "status": status if status is not None else ("bad_json" if exists else "missing"),
        "json_error": err,
        "ranked_csv_exists": ranked_exists,
        "top_csv_exists": top_exists,
        "device_csv_exists": device_exists,
        "ranked_csv": ranked_csv,
        "top_csv": top_csv,
        "device_csv": device_csv,
    })

check_df = pd.DataFrame(check_rows)

print("Expected experiments:", len(check_df))
print("Existing JSON:", int(check_df["json_exists"].sum()))
print("Missing JSON:", int((~check_df["json_exists"]).sum()))
print("OK JSON:", int((check_df["status"] == "ok").sum()))
print("Non-OK/Missing:", int((check_df["status"] != "ok").sum()))

print("\nCompletion by stage:")
display(
    check_df
    .groupby(["stage", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

print("\nCompletion by dataset:")
display(
    check_df
    .groupby(["stage", "dataset", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

missing_or_bad = check_df[check_df["status"] != "ok"].copy()

print("\nMissing / bad experiments:")
if missing_or_bad.empty:
    print("Everything expected is OK.")
else:
    display(missing_or_bad[[
        "stage", "dataset", "group", "contamination",
        "status", "expected_json", "json_error"
    ]].head(200))

# ------------------------------------------------------------
# CSV result checks
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("CSV OUTPUT CHECKS")
print("=" * 140)

csv_files = sorted(CSV_DIR.glob("*.csv")) if CSV_DIR.exists() else []

csv_rows = []
for p in csv_files:
    df_head, err = safe_read_csv_head(p, nrows=3)
    csv_rows.append({
        "file": p.name,
        "path": str(p),
        "size_mb": round(size_mb(p), 2),
        "read_ok": err is None,
        "read_error": err,
        "sample_rows": None if df_head is None else len(df_head),
        "sample_cols": None if df_head is None else df_head.shape[1],
        "columns_preview": None if df_head is None else list(df_head.columns[:20]),
    })

csv_df = pd.DataFrame(csv_rows)

print("CSV files:", len(csv_df))
if not csv_df.empty:
    display(csv_df.sort_values("size_mb", ascending=False).head(50))

    print("\nCSV type counts:")
    csv_df["kind"] = csv_df["file"].apply(
        lambda x:
        "agg_ranked" if x.startswith("agg__") and "__ranked" in x else
        "agg_top" if x.startswith("agg__") and "__top" in x else
        "ts_ranked_days" if x.startswith("ts__") and "__ranked_kkt_days" in x else
        "ts_top_days" if x.startswith("ts__") and "__top_kkt_days" in x else
        "ts_device_scores" if x.startswith("ts__") and "__device_aggregated_scores" in x else
        "other"
    )
    display(
        csv_df
        .groupby("kind")
        .agg(n_files=("file", "count"), total_mb=("size_mb", "sum"), max_mb=("size_mb", "max"))
        .reset_index()
        .sort_values("total_mb", ascending=False)
    )

# ------------------------------------------------------------
# Cache checks
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("CACHE CHECKS")
print("=" * 140)

cache_files = sorted(CACHE_DIR.glob("*")) if CACHE_DIR.exists() else []

cache_rows = []
for p in cache_files:
    if p.is_file():
        cache_rows.append({
            "file": p.name,
            "path": str(p),
            "suffix": p.suffix.lower(),
            "size_mb": round(size_mb(p), 2),
        })

cache_df = pd.DataFrame(cache_rows)

if cache_df.empty:
    print("No cache files found.")
else:
    display(cache_df.sort_values("size_mb", ascending=False))

# ------------------------------------------------------------
# Summary files checks
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("SUMMARY FILES CHECK")
print("=" * 140)

important_summary_files = [
    "all_kkt_anomaly_runs_summary.csv",
    "ablation_feature_group_run_summary.csv",
    "all_top_agg_kkt_anomalies_raw.csv",
    "final_agg_kkt_anomaly_ranking.csv",
    "all_top_ts_kkt_day_anomalies_raw.csv",
    "final_ts_kkt_day_anomaly_ranking.csv",
    "final_ts_device_anomaly_ranking.csv",
    "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv",
    "topk_overlap_with_all_numeric.csv",
    "KKT_ANOMALY_DETECTION_REPORT.txt",
]

summary_rows = []

for fname in important_summary_files:
    p = SUMMARY_DIR / fname
    row = {
        "file": fname,
        "exists": p.exists(),
        "size_mb": round(size_mb(p), 2) if p.exists() else 0.0,
        "path": str(p),
    }

    if p.exists() and p.suffix.lower() == ".csv":
        head, err = safe_read_csv_head(p, nrows=3)
        row["read_ok"] = err is None
        row["read_error"] = err
        row["sample_cols"] = None if head is None else head.shape[1]
        row["columns_preview"] = None if head is None else list(head.columns[:20])
    elif p.exists() and p.suffix.lower() == ".txt":
        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
            row["read_ok"] = True
            row["n_chars"] = len(text)
            row["preview"] = text[:500].replace("\n", " ")
        except Exception as e:
            row["read_ok"] = False
            row["read_error"] = repr(e)

    summary_rows.append(row)

summary_check_df = pd.DataFrame(summary_rows)
display(summary_check_df)

# ------------------------------------------------------------
# Show final top anomalies if available
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("FINAL TOP ANOMALIES PREVIEW")
print("=" * 140)

final_combined = SUMMARY_DIR / "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv"
final_agg = SUMMARY_DIR / "final_agg_kkt_anomaly_ranking.csv"
final_days = SUMMARY_DIR / "final_ts_kkt_day_anomaly_ranking.csv"
final_ts_device = SUMMARY_DIR / "final_ts_device_anomaly_ranking.csv"

if final_combined.exists():
    print("\nFINAL_COMBINED_KKT_ANOMALY_RANKING.csv")
    df = pd.read_csv(final_combined)
    print("shape:", df.shape)
    display(df.head(30))
elif final_agg.exists() or final_ts_device.exists():
    print("Combined final file is missing, but partial final rankings exist.")

    if final_agg.exists():
        print("\nfinal_agg_kkt_anomaly_ranking.csv")
        df = pd.read_csv(final_agg)
        print("shape:", df.shape)
        display(df.head(20))

    if final_ts_device.exists():
        print("\nfinal_ts_device_anomaly_ranking.csv")
        df = pd.read_csv(final_ts_device)
        print("shape:", df.shape)
        display(df.sort_values("ts_final_score", ascending=False).head(20))
else:
    print("No final KKT-level anomaly ranking found yet.")

if final_days.exists():
    print("\nfinal_ts_kkt_day_anomaly_ranking.csv")
    df = pd.read_csv(final_days)
    print("shape:", df.shape)
    display(df.head(30))
else:
    print("\nNo final KKT-day anomaly ranking found yet.")

# ------------------------------------------------------------
# What to rerun
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("WHAT TO RERUN")
print("=" * 140)

if missing_or_bad.empty:
    print("All expected experiment JSON files are OK.")
else:
    print("These exact experiments are missing or failed:")
    display(missing_or_bad[[
        "stage", "dataset", "group", "contamination", "status", "expected_json"
    ]])

    print("\nTo continue:")
    print("1. Do NOT delete outputs.")
    print("2. Set FORCE = False.")
    print("3. Rerun the original pipeline cells from the experiment loops.")
    print("4. Existing JSON files will be skipped, missing ones will be computed.")

# ------------------------------------------------------------
# Save this audit
# ------------------------------------------------------------

AUDIT_OUT = SUMMARY_DIR / "PIPELINE_COMPLETION_AUDIT"

AUDIT_OUT.mkdir(parents=True, exist_ok=True)

dir_df.to_csv(AUDIT_OUT / "directory_summary.csv", index=False, encoding="utf-8-sig")
json_df.to_csv(AUDIT_OUT / "json_inventory.csv", index=False, encoding="utf-8-sig")
check_df.to_csv(AUDIT_OUT / "expected_vs_actual_experiments.csv", index=False, encoding="utf-8-sig")
csv_df.to_csv(AUDIT_OUT / "csv_inventory.csv", index=False, encoding="utf-8-sig")
cache_df.to_csv(AUDIT_OUT / "cache_inventory.csv", index=False, encoding="utf-8-sig")
summary_check_df.to_csv(AUDIT_OUT / "summary_files_check.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("AUDIT SAVED")
print("=" * 140)
print(AUDIT_OUT)
print("Saved:")
print(" - directory_summary.csv")
print(" - json_inventory.csv")
print(" - expected_vs_actual_experiments.csv")
print(" - csv_inventory.csv")
print(" - cache_inventory.csv")
print(" - summary_files_check.csv")

print("\nDONE.")

KKT ANOMALY PIPELINE COMPLETION CHECK
Time: 2026-05-17 22:19:21
OUT_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs
OUT_ROOT exists: True
Expected agg experiments: 66
Expected ts experiments: 42
Expected total experiments: 108

DIRECTORY SUMMARY


,folder,exists,files,size_mb,path
0,logs,True,115,0.23,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
1,results_json,True,108,0.25,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
2,results_csv,True,210,43516.57,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
3,summary,True,17,139.05,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
4,cache,True,2,4288.21,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...



JSON RESULT INVENTORY
JSON files: 108


,file,path,status,error,stage,dataset,group,contamination,n_rows,n_devices,n_features,n_top_anomalies,n_top_anomaly_days,runtime_sec,ranked_csv,top_csv,device_csv
0,agg__agg_kkt_47_11_all_numeric_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,all_numeric,0.01,34443,NaN,150,345.0,NaN,21.924367,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
1,agg__agg_kkt_47_11_all_numeric_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,all_numeric,0.03,34443,NaN,150,1034.0,NaN,18.555602,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
2,agg__agg_kkt_47_11_all_numeric_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,all_numeric,0.05,34443,NaN,150,1723.0,NaN,19.884701,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
3,agg__agg_kkt_47_11_benford_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,None,agg_kkt_47_11,benford,0.01,34443,NaN,0,NaN,NaN,NaN,None,None,None
4,agg__agg_kkt_47_11_benford_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,None,agg_kkt_47_11,benford,0.03,34443,NaN,0,NaN,NaN,NaN,None,None,None
5,agg__agg_kkt_47_11_benford_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,None,agg_kkt_47_11,benford,0.05,34443,NaN,0,NaN,NaN,NaN,None,None,None
6,agg__agg_kkt_47_11_bills_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,bills,0.01,34443,NaN,11,345.0,NaN,24.627632,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
7,agg__agg_kkt_47_11_bills_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,bills,0.03,34443,NaN,11,1034.0,NaN,26.645549,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
8,agg__agg_kkt_47_11_bills_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,ok,None,agg_kkt_level,agg_kkt_47_11,bills,0.05,34443,NaN,11,1723.0,NaN,26.971250,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None
9,agg__agg_kkt_47_11_cross_corr_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,skipped_no_features,None,None,agg_kkt_47_11,cross_corr,0.01,34443,NaN,0,NaN,NaN,NaN,None,None,None



Status counts:


...


final_ts_kkt_day_anomaly_ranking.csv
shape: (6791, 15)


,kkt_id,date,trade_store_hash,trade_object_hash,n_appearances,best_day_score,mean_day_score,total_sum,n_bills,total_ecash,ecash_fraction,explanation,source_files,final_day_anomaly_score,final_day_rank
0,a32f067711de2e6a4eab509c3590f090,2023-12-31,x72df01080c58dc3aa91e6cd9c9400aa64badb12bfa9f9...,x174c4cadcad18eb2e89033ac22babb590f39533dce3df...,6,1.0,1.0,1287178.40,413.0,1081986.40,0.840588,total_cash=14.30; total_sum=12.38; total_ecash...,ts__ts_kkt_47_11_core_daily_contam_0.01__top_k...,1.0,1
1,dc978bcbb6686f0e70c173d2367e131a,2023-12-26,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,94620.00,1.0,0.00,0.000000,min_sum=1819.45; avg_price_piece=436.98; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,2
2,dc978bcbb6686f0e70c173d2367e131a,2023-12-18,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,101982.00,2.0,0.00,0.000000,avg_price_piece=235.29; min_sum=197.95; avg_pr...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,3
3,dc978bcbb6686f0e70c173d2367e131a,2023-12-19,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,92560.00,1.0,0.00,0.000000,min_sum=1779.84; avg_price_piece=427.45; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,4
4,45b5abf828e4410557284e89ab1223b8,2023-12-28,xfe3c5105cd73db6bf3265e3c81af777c08cfae0d735e2...,xf73ece543d2d107e1f8d56f1d34a28c687d455aa1066d...,6,1.0,1.0,94470.00,4.0,94470.00,1.000000,total_sum__roll7_rel_jump=240227123200.00; tot...,ts__ts_kkt_47_19_all_numeric_contam_0.01__top_...,1.0,5
5,dc978bcbb6686f0e70c173d2367e131a,2023-12-20,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,210055.00,3.0,0.00,0.000000,min_sum=432.53; avg_price_piece=323.25; avg_pr...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,6
6,45d06c5a6c5dcf992c963f852a099cf0,2023-11-10,x00ce955f7367d0b8bb54eeefdce510107c1f811db81eb...,xe24a6edbbc6f436e3207a34e4334c014fcb5333477b80...,6,1.0,1.0,2952.00,8.0,2734.00,0.926152,total_sum__roll7_rel_jump=7506620928.00; total...,ts__ts_kkt_47_19_all_numeric_contam_0.01__top_...,1.0,7
7,dc978bcbb6686f0e70c173d2367e131a,2023-12-22,x0afc1f0dbcba157d3959652f18dc8f39a86cd162250e6...,xf3fe363d4d2f0f98f283b903f4e7ac0475da6e25e906d...,6,1.0,1.0,191560.00,2.0,0.00,0.000000,min_sum=1841.76; avg_price_piece=442.34; avg_p...,ts__ts_kkt_47_19_core_daily_contam_0.01__top_k...,1.0,8
8,45d3a561ece2a6c6bbfa2712c2ebbf7b,2023-10-31,x3f470cdd629b840b2d2fa4a0af79030699f604c53cc8f...,x835f45c964f3f44e85e0e733d15ffdd3aca5323087a5c...,6,1.0,1.0,914605.00,1.0,0.00,0.000000,derived_cash_to_ecash=2477367469539328.00; avg...,ts__ts_kkt_47_19_all_numeric_contam_0.01__top_...,1.0,9
9,45d3a561ece2a6c6bbfa2712c2ebbf7b,2023-11-09,x3f470cdd629b840b2d2fa4a0af79030699f604c53cc8f...,x835f45c964f3f44e85e0e733d15ffdd3aca5323087a5c...,6,1.0,1.0,400000.00,1.0,0.00,0.000000,derived_cash_to_ecash=1083469857816576.00; avg...,ts__ts_kkt_47_19_all_numeric_contam_0.01__top_...,1.0,10



WHAT TO RERUN
These exact experiments are missing or failed:


,stage,dataset,group,contamination,status,expected_json
15,agg_kkt_level,agg_kkt_47_11,temporal,0.01,skipped_no_features,agg__agg_kkt_47_11_temporal_contam_0.01.json
16,agg_kkt_level,agg_kkt_47_11,temporal,0.03,skipped_no_features,agg__agg_kkt_47_11_temporal_contam_0.03.json
17,agg_kkt_level,agg_kkt_47_11,temporal,0.05,skipped_no_features,agg__agg_kkt_47_11_temporal_contam_0.05.json
18,agg_kkt_level,agg_kkt_47_11,benford,0.01,skipped_no_features,agg__agg_kkt_47_11_benford_contam_0.01.json
19,agg_kkt_level,agg_kkt_47_11,benford,0.03,skipped_no_features,agg__agg_kkt_47_11_benford_contam_0.03.json
20,agg_kkt_level,agg_kkt_47_11,benford,0.05,skipped_no_features,agg__agg_kkt_47_11_benford_contam_0.05.json
21,agg_kkt_level,agg_kkt_47_11,cross_corr,0.01,skipped_no_features,agg__agg_kkt_47_11_cross_corr_contam_0.01.json
22,agg_kkt_level,agg_kkt_47_11,cross_corr,0.03,skipped_no_features,agg__agg_kkt_47_11_cross_corr_contam_0.03.json
23,agg_kkt_level,agg_kkt_47_11,cross_corr,0.05,skipped_no_features,agg__agg_kkt_47_11_cross_corr_contam_0.05.json
24,agg_kkt_level,agg_kkt_47_11,shape_complexity,0.01,skipped_no_features,agg__agg_kkt_47_11_shape_complexity_contam_0.0...



To continue:
1. Do NOT delete outputs.
2. Set FORCE = False.
3. Rerun the original pipeline cells from the experiment loops.
4. Existing JSON files will be skipped, missing ones will be computed.

AUDIT SAVED
C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\PIPELINE_COMPLETION_AUDIT
Saved:
 - directory_summary.csv
 - json_inventory.csv
 - expected_vs_actual_experiments.csv
 - csv_inventory.csv
 - cache_inventory.csv
 - summary_files_check.csv

DONE.


In [2]:
from pathlib import Path
import json
import pandas as pd

OUT_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs")
JSON_DIR = OUT_ROOT / "results_json"

rows = []

for p in sorted(JSON_DIR.glob("*.json")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)
    except Exception as e:
        rows.append({
            "file": p.name,
            "status": "bad_json",
            "error": repr(e),
        })
        continue

    rows.append({
        "file": p.name,
        "stage": r.get("stage"),
        "dataset": r.get("dataset"),
        "group": r.get("group"),
        "contamination": r.get("contamination"),
        "status": r.get("status"),
        "n_features": r.get("n_features"),
        "error": r.get("error"),
    })

df = pd.DataFrame(rows)

print("Status counts:")
display(df.groupby(["stage", "dataset", "status"], dropna=False).size().reset_index(name="count"))

print("Skipped:")
display(df[df["status"] == "skipped_no_features"])

Status counts:


,stage,dataset,status,count
0,agg_kkt_level,agg_kkt_47_11,ok,21
1,agg_kkt_level,agg_kkt_47_19,ok,21
2,ts_kkt_day_level,ts_kkt_47_11,ok,21
3,ts_kkt_day_level,ts_kkt_47_19,ok,21
4,NaN,agg_kkt_47_11,skipped_no_features,12
5,NaN,agg_kkt_47_19,skipped_no_features,12


Skipped:


,file,stage,dataset,group,contamination,status,n_features,error
3,agg__agg_kkt_47_11_benford_contam_0.01.json,None,agg_kkt_47_11,benford,0.01,skipped_no_features,0,None
4,agg__agg_kkt_47_11_benford_contam_0.03.json,None,agg_kkt_47_11,benford,0.03,skipped_no_features,0,None
5,agg__agg_kkt_47_11_benford_contam_0.05.json,None,agg_kkt_47_11,benford,0.05,skipped_no_features,0,None
9,agg__agg_kkt_47_11_cross_corr_contam_0.01.json,None,agg_kkt_47_11,cross_corr,0.01,skipped_no_features,0,None
10,agg__agg_kkt_47_11_cross_corr_contam_0.03.json,None,agg_kkt_47_11,cross_corr,0.03,skipped_no_features,0,None
11,agg__agg_kkt_47_11_cross_corr_contam_0.05.json,None,agg_kkt_47_11,cross_corr,0.05,skipped_no_features,0,None
21,agg__agg_kkt_47_11_shape_complexity_contam_0.0...,None,agg_kkt_47_11,shape_complexity,0.01,skipped_no_features,0,None
22,agg__agg_kkt_47_11_shape_complexity_contam_0.0...,None,agg_kkt_47_11,shape_complexity,0.03,skipped_no_features,0,None
23,agg__agg_kkt_47_11_shape_complexity_contam_0.0...,None,agg_kkt_47_11,shape_complexity,0.05,skipped_no_features,0,None
27,agg__agg_kkt_47_11_temporal_contam_0.01.json,None,agg_kkt_47_11,temporal,0.01,skipped_no_features,0,None


In [3]:
# ============================================================
# FIX aggregate feature-group parser
# ============================================================

ID_COLS = [
    "kkt_id",
    "trade_store_hash",
    "trade_object_hash",
    "region",
    "okved2",
    "okved_group",
    "type",
    "category",
    "date",
]

FEATURE_GROUPS = [
    "all_numeric",
    "turnover",
    "bills",
    "items_price",
    "payment_mix",
    "temporal",
    "benford",
    "cross_corr",
    "shape_complexity",
    "non_temporal",
    "simple_core",
]

SIMPLE_CORE_COL_HINTS = [
    "total_sum",
    "avg_sum",
    "max_sum",
    "min_sum",
    "median_sum",
    "std_sum",
    "total_cash",
    "total_ecash",
    "n_bills",
    "n_items",
    "total_items",
    "ecash_fraction",
    "cash_fraction",
    "avg_price_item",
    "avg_price_piece",
    "avg_cash",
    "avg_ecash",
]

def infer_feature_group(col: str) -> str:
    c = str(col).lower()

    if c in ID_COLS:
        return "id"

    # ВАЖНО: сначала проверяем специальные suffix/family признаки,
    # а уже потом базовые переменные типа total_sum.
    if "benford" in c:
        return "benford"

    if (
        "autocorrelation" in c
        or "partial_autocorrelation" in c
        or "trend" in c
        or "seasonal" in c
        or "frequency" in c
        or "fft" in c
        or "phase" in c
    ):
        return "temporal"

    if "corr" in c:
        return "cross_corr"

    if (
        "ratio_beyond" in c
        or "number_peaks" in c
        or "longest_strike" in c
        or "cid_ce" in c
        or "mean_abs_change" in c
        or "absolute_sum_of_changes" in c
        or "variation_coefficient" in c
        or "skewness" in c
        or "skew" in c
    ):
        return "shape_complexity"

    if (
        "total_sum" in c
        or "avg_sum" in c
        or "max_sum" in c
        or "min_sum" in c
        or "median_sum" in c
        or "std_sum" in c
    ):
        return "turnover"

    if "bill" in c or "n_bills" in c:
        return "bills"

    if "item" in c or "price" in c or "piece" in c:
        return "items_price"

    if "cash" in c or "ecash" in c or "payment" in c:
        return "payment_mix"

    return "other_numeric"


def get_numeric_feature_columns(df):
    cols = []
    for c in df.columns:
        if c in ID_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def select_columns_by_group(df, group: str):
    numeric_cols = get_numeric_feature_columns(df)

    if group == "all_numeric":
        return numeric_cols

    if group == "non_temporal":
        return [c for c in numeric_cols if infer_feature_group(c) != "temporal"]

    if group == "simple_core":
        selected = []
        for c in numeric_cols:
            cl = str(c).lower()
            if any(h in cl for h in SIMPLE_CORE_COL_HINTS):
                selected.append(c)
        return selected

    return [c for c in numeric_cols if infer_feature_group(c) == group]


def describe_feature_groups(df):
    rows = []
    for group in FEATURE_GROUPS:
        cols = select_columns_by_group(df, group)
        rows.append({
            "group": group,
            "n_cols": len(cols),
            "columns_preview": cols[:20],
        })
    return pd.DataFrame(rows)

In [4]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2")

AGG_PATHS = {
    "agg_kkt_47_11": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.11_region_77_impute_True.csv",
    "agg_kkt_47_19": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.19_region_77_impute_True.csv",
}

for name, path in AGG_PATHS.items():
    print("\n" + "=" * 120)
    print(name)
    print("=" * 120)

    df_agg = pd.read_csv(path)
    group_df = describe_feature_groups(df_agg)
    display(group_df)

    empty_groups = group_df[group_df["n_cols"] == 0]["group"].tolist()
    print("Empty groups:", empty_groups)


agg_kkt_47_11


,group,n_cols,columns_preview
0,all_numeric,149,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
1,turnover,30,"[avg_sum__mean, avg_sum__standard_deviation, m..."
2,bills,5,"[ecash_bills_fraction__mean, ecash_bills_fract..."
3,items_price,8,"[avg_price_item__mean, avg_price_item__standar..."
4,payment_mix,15,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
5,temporal,38,[ecash_bills_fraction__agg_linear_trend__attr_...
6,benford,1,[total_sum__benford_correlation]
7,cross_corr,13,"[total_sum__n_bills__corr, total_sum__avg_sum_..."
8,shape_complexity,38,"[avg_sum__skewness, avg_sum__ratio_beyond_r_si..."
9,non_temporal,111,"[avg_ecash__mean, avg_ecash__standard_deviatio..."


Empty groups: []

agg_kkt_47_19


,group,n_cols,columns_preview
0,all_numeric,149,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
1,turnover,30,"[avg_sum__mean, avg_sum__standard_deviation, m..."
2,bills,5,"[ecash_bills_fraction__mean, ecash_bills_fract..."
3,items_price,8,"[avg_price_item__mean, avg_price_item__standar..."
4,payment_mix,15,"[avg_ecash__mean, avg_ecash__standard_deviatio..."
5,temporal,38,[ecash_bills_fraction__agg_linear_trend__attr_...
6,benford,1,[total_sum__benford_correlation]
7,cross_corr,13,"[total_sum__n_bills__corr, total_sum__avg_sum_..."
8,shape_complexity,38,"[avg_sum__skewness, avg_sum__ratio_beyond_r_si..."
9,non_temporal,111,"[avg_ecash__mean, avg_ecash__standard_deviatio..."


Empty groups: []


In [5]:
from pathlib import Path
import json

OUT_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs")
JSON_DIR = OUT_ROOT / "results_json"

removed = []

for p in sorted(JSON_DIR.glob("*.json")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)
    except Exception:
        continue

    if r.get("status") == "skipped_no_features":
        p.unlink()
        removed.append(p.name)

print("Removed skipped_no_features JSON:", len(removed))
for x in removed:
    print(" -", x)

Removed skipped_no_features JSON: 24
 - agg__agg_kkt_47_11_benford_contam_0.01.json
 - agg__agg_kkt_47_11_benford_contam_0.03.json
 - agg__agg_kkt_47_11_benford_contam_0.05.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.01.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.03.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.05.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.01.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.03.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.05.json
 - agg__agg_kkt_47_11_temporal_contam_0.01.json
 - agg__agg_kkt_47_11_temporal_contam_0.03.json
 - agg__agg_kkt_47_11_temporal_contam_0.05.json
 - agg__agg_kkt_47_19_benford_contam_0.01.json
 - agg__agg_kkt_47_19_benford_contam_0.03.json
 - agg__agg_kkt_47_19_benford_contam_0.05.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.01.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.03.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.05.json
 - agg__agg_kkt_47_19_shape_complexity_contam_0.01.json
 - agg__agg_k

In [18]:
# ============================================================
# SELF-CONTAINED FIX:
# Recompute only aggregate skipped feature groups:
# temporal / benford / cross_corr / shape_complexity
#
# Does not rely on old select_columns_by_group().
# Saves results into existing kkt_anomaly_outputs/results_json and results_csv.
# ============================================================

from pathlib import Path
import json
import time
from datetime import datetime
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_data\test_pipeline_2025_2")
OUT_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs")

JSON_DIR = OUT_ROOT / "results_json"
CSV_DIR = OUT_ROOT / "results_csv"
LOG_DIR = OUT_ROOT / "logs"
SUMMARY_DIR = OUT_ROOT / "summary"

for d in [JSON_DIR, CSV_DIR, LOG_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

AGG_PATHS = {
    "agg_kkt_47_11": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.11_region_77_impute_True.csv",
    "agg_kkt_47_19": DATA_ROOT / "dataloader_aggregated" / "agg_kkt_okved_47.19_region_77_impute_True.csv",
}

TARGET_GROUPS = [
    "temporal",
    "benford",
    "cross_corr",
    "shape_complexity",
]

CONTAMINATION_LEVELS = [0.01, 0.03, 0.05]

ID_COLS = {
    "kkt_id",
    "trade_store_hash",
    "trade_object_hash",
    "region",
    "okved2",
    "okved_group",
    "type",
    "category",
    "date",
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def safe_name(x):
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def log_line(path, msg, also_print=True):
    line = f"[{now_str()}] {msg}"
    if also_print:
        print(line)
    with open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_json(path, obj):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_score(s):
    s = np.asarray(s, dtype=float)
    finite = np.isfinite(s)
    if not finite.any():
        return np.zeros_like(s, dtype=float)
    med = np.nanmedian(s[finite])
    s = np.nan_to_num(s, nan=med, posinf=np.nanmax(s[finite]), neginf=np.nanmin(s[finite]))
    lo, hi = np.percentile(s, [1, 99])
    if hi <= lo:
        lo, hi = np.min(s), np.max(s)
    if hi <= lo:
        return np.zeros_like(s, dtype=float)
    return np.clip((s - lo) / (hi - lo), 0, 1)


def prepare_matrix(df, feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    Xs = pipe.fit_transform(X)
    Xs = np.asarray(Xs, dtype=np.float32)
    Xs = np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)
    return Xs


def robust_z_score_anomaly(X):
    return normalize_score(np.sqrt(np.mean(X ** 2, axis=1)))


def pca_reconstruction_anomaly(X):
    n, d = X.shape
    if d <= 1 or n <= 2:
        return robust_z_score_anomaly(X)
    n_components = min(10, max(1, d // 3), d - 1, n - 1)
    if n_components < 1:
        return robust_z_score_anomaly(X)
    pca = PCA(n_components=n_components, random_state=SEED)
    Z = pca.fit_transform(X)
    Xrec = pca.inverse_transform(Z)
    err = np.mean((X - Xrec) ** 2, axis=1)
    return normalize_score(err)


def isolation_forest_anomaly(X, contamination):
    model = IsolationForest(
        n_estimators=400,
        contamination=contamination,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X)
    return normalize_score(-model.decision_function(X))


def lof_anomaly(X, contamination, max_n=120_000):
    if X.shape[0] > max_n:
        return None
    model = LocalOutlierFactor(
        n_neighbors=35,
        contamination=contamination,
        novelty=False,
        n_jobs=-1,
    )
    model.fit_predict(X)
    return normalize_score(-model.negative_outlier_factor_)


def elliptic_anomaly(X, contamination, max_n=100_000, max_d=80):
    n, d = X.shape
    if n > max_n or d > max_d or n <= d + 10:
        return None
    model = EllipticEnvelope(
        contamination=contamination,
        random_state=SEED,
    )
    model.fit(X)
    return normalize_score(-model.decision_function(X))


def compute_scores(X, contamination):
    scores = {}
    scores["robust_z"] = robust_z_score_anomaly(X)
    scores["pca_reconstruction"] = pca_reconstruction_anomaly(X)
    scores["isolation_forest"] = isolation_forest_anomaly(X, contamination)

    lof = lof_anomaly(X, contamination)
    if lof is not None:
        scores["lof"] = lof

    ell = elliptic_anomaly(X, contamination)
    if ell is not None:
        scores["elliptic_envelope"] = ell

    mat = np.vstack([normalize_score(v) for v in scores.values()]).T
    scores["consensus_mean"] = np.mean(mat, axis=1)
    scores["consensus_max"] = np.max(mat, axis=1)
    return scores


def top_feature_contributions(X_scaled, feature_cols, row_idx, top_n=10):
    row = X_scaled[row_idx]
    abs_vals = np.abs(row)
    order = np.argsort(-abs_vals)[:top_n]
    return "; ".join([f"{feature_cols[j]}={row[j]:.2f}" for j in order])


def select_special_group_columns(df, group):
    numeric_cols = [
        c for c in df.columns
        if c not in ID_COLS and pd.api.types.is_numeric_dtype(df[c])
    ]

    out = []

    for c in numeric_cols:
        cl = str(c).lower()

        if group == "benford":
            if "benford" in cl:
                out.append(c)

        elif group == "temporal":
            if (
                "autocorrelation" in cl
                or "partial_autocorrelation" in cl
                or "trend" in cl
                or "seasonal" in cl
                or "frequency" in cl
                or "fft" in cl
                or "phase" in cl
            ):
                out.append(c)

        elif group == "cross_corr":
            if "corr" in cl and "autocorr" not in cl and "autocorrelation" not in cl:
                out.append(c)

        elif group == "shape_complexity":
            if (
                "ratio_beyond" in cl
                or "number_peaks" in cl
                or "longest_strike" in cl
                or "cid_ce" in cl
                or "mean_abs_change" in cl
                or "absolute_sum_of_changes" in cl
                or "variation_coefficient" in cl
                or "skewness" in cl
            ):
                out.append(c)

        else:
            raise ValueError(group)

    return out


def expected_json_path(dataset_name, group, contamination):
    run_id = f"{dataset_name}__{group}__contam_{contamination}"
    return JSON_DIR / f"agg__{safe_name(run_id)}.json"


def expected_csv_paths(dataset_name, group, contamination):
    run_id = f"{dataset_name}__{group}__contam_{contamination}"
    ranked = CSV_DIR / f"agg__{safe_name(run_id)}__ranked.csv"
    top = CSV_DIR / f"agg__{safe_name(run_id)}__top_anomalies.csv"
    log = LOG_DIR / f"agg__{safe_name(run_id)}.txt"
    return ranked, top, log


# ------------------------------------------------------------
# 1. Remove skipped JSON only for target groups
# ------------------------------------------------------------

removed = []

for p in sorted(JSON_DIR.glob("agg__*.json")):
    try:
        r = load_json(p)
    except Exception:
        continue

    if r.get("status") == "skipped_no_features" and r.get("group") in TARGET_GROUPS:
        p.unlink()
        removed.append(p.name)

print("Removed skipped target JSON:", len(removed))
for x in removed:
    print(" -", x)

# ------------------------------------------------------------
# 2. Recompute target groups
# ------------------------------------------------------------

for dataset_name, path in AGG_PATHS.items():
    print("\n" + "=" * 120)
    print("DATASET:", dataset_name)
    print("PATH:", path)
    print("=" * 120)

    df = pd.read_csv(path)
    df["_row_id"] = np.arange(len(df))
    df["_source_dataset"] = dataset_name

    print("shape:", df.shape)

    for group in TARGET_GROUPS:
        feature_cols = select_special_group_columns(df, group)

        print(f"\nGROUP {group}: n_features={len(feature_cols)}")
        print("preview:", feature_cols[:20])

        if len(feature_cols) == 0:
            print("[WARNING] still empty, skipping:", dataset_name, group)
            continue

        X_scaled = prepare_matrix(df, feature_cols)

        for contamination in CONTAMINATION_LEVELS:
            json_path = expected_json_path(dataset_name, group, contamination)
            ranked_csv, top_csv, log_path = expected_csv_paths(dataset_name, group, contamination)

            if json_path.exists():
                try:
                    r = load_json(json_path)
                    if r.get("status") == "ok":
                        print("[SKIP OK]", json_path.name)
                        continue
                except Exception:
                    pass

            t0 = time.time()

            log_line(log_path, "=" * 120)
            log_line(log_path, f"RUN FIXED AGG dataset={dataset_name}, group={group}, contamination={contamination}")
            log_line(log_path, "=" * 120)
            log_line(log_path, f"n_rows: {len(df)}")
            log_line(log_path, f"n_features: {len(feature_cols)}")
            log_line(log_path, f"feature preview: {feature_cols[:30]}")

            scores = compute_scores(X_scaled, contamination)

            out = df.copy()

            for model_name, s in scores.items():
                out[f"score_{model_name}"] = normalize_score(s)

            out["anomaly_score"] = out["score_consensus_mean"]
            out["anomaly_rank"] = out["anomaly_score"].rank(ascending=False, method="first").astype(int)
            out["anomaly_percentile"] = out["anomaly_score"].rank(pct=True)

            threshold = np.quantile(out["anomaly_score"], 1.0 - contamination)
            out["is_top_anomaly"] = out["anomaly_score"] >= threshold

            out = out.sort_values("anomaly_score", ascending=False).reset_index(drop=True)

            top_n = min(500, max(50, int(len(out) * max(contamination, 0.01))))
            top_df = out.head(top_n).copy()

            explanations = []
            for _, row in top_df.iterrows():
                row_id = int(row["_row_id"])
                explanations.append(top_feature_contributions(X_scaled, feature_cols, row_id, top_n=10))

            top_df["top_feature_explanation"] = explanations
            top_df["top_feature_groups"] = group

            out.to_csv(ranked_csv, index=False, encoding="utf-8-sig")
            top_df.to_csv(top_csv, index=False, encoding="utf-8-sig")

            runtime = time.time() - t0

            result = {
                "status": "ok",
                "stage": "agg_kkt_level",
                "dataset": dataset_name,
                "group": group,
                "contamination": contamination,
                "n_rows": int(len(df)),
                "n_features": int(len(feature_cols)),
                "feature_cols": feature_cols,
                "score_columns": list(scores.keys()),
                "threshold": float(threshold),
                "n_top_anomalies": int(out["is_top_anomaly"].sum()),
                "top_score": float(out["anomaly_score"].max()),
                "median_score": float(out["anomaly_score"].median()),
                "runtime_sec": float(runtime),
                "ranked_csv": str(ranked_csv),
                "top_csv": str(top_csv),
                "created_at": now_str(),
            }

            save_json(json_path, result)

            log_line(log_path, f"threshold: {threshold:.6f}")
            log_line(log_path, f"n_top_anomalies: {int(out['is_top_anomaly'].sum())}")
            log_line(log_path, f"top_score: {out['anomaly_score'].max():.6f}")
            log_line(log_path, f"Saved ranked: {ranked_csv}")
            log_line(log_path, f"Saved top: {top_csv}")
            log_line(log_path, f"Runtime sec: {runtime:.2f}")

            print("[DONE]", dataset_name, group, contamination, "runtime", round(runtime, 2))

# ------------------------------------------------------------
# 3. Verify status
# ------------------------------------------------------------

rows = []

for p in sorted(JSON_DIR.glob("*.json")):
    try:
        r = load_json(p)
    except Exception as e:
        rows.append({"file": p.name, "status": "bad_json", "error": repr(e)})
        continue

    rows.append({
        "file": p.name,
        "stage": r.get("stage"),
        "dataset": r.get("dataset"),
        "group": r.get("group"),
        "contamination": r.get("contamination"),
        "status": r.get("status"),
        "n_features": r.get("n_features"),
    })

status_df = pd.DataFrame(rows)

print("\n" + "=" * 120)
print("STATUS AFTER FIXED RECOMPUTE")
print("=" * 120)

display(
    status_df
    .groupby(["stage", "dataset", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

non_ok = status_df[status_df["status"] != "ok"]
print("Non-OK:", len(non_ok))
display(non_ok.sort_values(["dataset", "group", "contamination"]).head(100))

Removed skipped target JSON: 24
 - agg__agg_kkt_47_11_benford_contam_0.01.json
 - agg__agg_kkt_47_11_benford_contam_0.03.json
 - agg__agg_kkt_47_11_benford_contam_0.05.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.01.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.03.json
 - agg__agg_kkt_47_11_cross_corr_contam_0.05.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.01.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.03.json
 - agg__agg_kkt_47_11_shape_complexity_contam_0.05.json
 - agg__agg_kkt_47_11_temporal_contam_0.01.json
 - agg__agg_kkt_47_11_temporal_contam_0.03.json
 - agg__agg_kkt_47_11_temporal_contam_0.05.json
 - agg__agg_kkt_47_19_benford_contam_0.01.json
 - agg__agg_kkt_47_19_benford_contam_0.03.json
 - agg__agg_kkt_47_19_benford_contam_0.05.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.01.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.03.json
 - agg__agg_kkt_47_19_cross_corr_contam_0.05.json
 - agg__agg_kkt_47_19_shape_complexity_contam_0.01.json
 - agg__agg_kkt_47

,stage,dataset,status,count
0,agg_kkt_level,agg_kkt_47_11,ok,33
1,agg_kkt_level,agg_kkt_47_19,ok,33
2,ts_kkt_day_level,ts_kkt_47_11,ok,21
3,ts_kkt_day_level,ts_kkt_47_19,ok,21


Non-OK: 0


,file,stage,dataset,group,contamination,status,n_features


In [20]:
# ============================================================
# FINAL ALL-IN-ONE STATISTICS CELL
# KKT anomaly pipeline final audit + tables + compact report
# ============================================================

from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

OUT_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs")

JSON_DIR = OUT_ROOT / "results_json"
CSV_DIR = OUT_ROOT / "results_csv"
SUMMARY_DIR = OUT_ROOT / "summary"
FINAL_STATS_DIR = SUMMARY_DIR / "FINAL_STATISTICS"

FINAL_STATS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 140)
print("FINAL KKT ANOMALY PIPELINE STATISTICS")
print("=" * 140)
print("Time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("OUT_ROOT:", OUT_ROOT)
print("JSON_DIR:", JSON_DIR)
print("CSV_DIR:", CSV_DIR)
print("SUMMARY_DIR:", SUMMARY_DIR)
print("FINAL_STATS_DIR:", FINAL_STATS_DIR)

# ------------------------------------------------------------
# Expected design
# ------------------------------------------------------------

AGG_DATASETS = ["agg_kkt_47_11", "agg_kkt_47_19"]
TS_DATASETS = ["ts_kkt_47_11", "ts_kkt_47_19"]

AGG_FEATURE_GROUPS = [
    "all_numeric",
    "turnover",
    "bills",
    "items_price",
    "payment_mix",
    "temporal",
    "benford",
    "cross_corr",
    "shape_complexity",
    "non_temporal",
    "simple_core",
]

TS_FEATURE_GROUPS = [
    "all_numeric",
    "core_daily",
    "rolling_deviation",
    "peer_deviation",
    "payment_mix",
    "turnover",
    "bills_items",
]

CONTAMINATION_LEVELS = [0.01, 0.03, 0.05]

EXPECTED_AGG = len(AGG_DATASETS) * len(AGG_FEATURE_GROUPS) * len(CONTAMINATION_LEVELS)
EXPECTED_TS = len(TS_DATASETS) * len(TS_FEATURE_GROUPS) * len(CONTAMINATION_LEVELS)
EXPECTED_TOTAL = EXPECTED_AGG + EXPECTED_TS

print("\nExpected experiments:")
print("  aggregate:", EXPECTED_AGG)
print("  time-series:", EXPECTED_TS)
print("  total:", EXPECTED_TOTAL)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_read_json(path: Path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, repr(e)


def safe_read_csv(path: Path, nrows=None):
    try:
        return pd.read_csv(path, nrows=nrows), None
    except Exception as e:
        return None, repr(e)


def size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    if path.is_file():
        return path.stat().st_size / 1024 / 1024
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / 1024 / 1024


def count_files(path: Path) -> int:
    if not path.exists():
        return 0
    if path.is_file():
        return 1
    return sum(1 for p in path.rglob("*") if p.is_file())


def safe_name(x):
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',', ';']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def expected_json_name(prefix, dataset, group, contamination):
    run_id = f"{dataset}__{group}__contam_{contamination}"
    return f"{prefix}__{safe_name(run_id)}.json"


def normalize_score(s):
    s = np.asarray(s, dtype=float)
    if len(s) == 0:
        return s
    finite = np.isfinite(s)
    if not finite.any():
        return np.zeros_like(s)
    med = np.nanmedian(s[finite])
    s = np.nan_to_num(s, nan=med, posinf=np.nanmax(s[finite]), neginf=np.nanmin(s[finite]))
    lo, hi = np.percentile(s, [1, 99])
    if hi <= lo:
        lo, hi = np.min(s), np.max(s)
    if hi <= lo:
        return np.zeros_like(s)
    return np.clip((s - lo) / (hi - lo), 0, 1)


# ------------------------------------------------------------
# 1. Directory inventory
# ------------------------------------------------------------

dir_rows = []

for d in [JSON_DIR, CSV_DIR, SUMMARY_DIR]:
    dir_rows.append({
        "folder": d.name,
        "exists": d.exists(),
        "n_files": count_files(d),
        "size_mb": round(size_mb(d), 3),
        "path": str(d),
    })

dir_df = pd.DataFrame(dir_rows)
dir_df.to_csv(FINAL_STATS_DIR / "directory_inventory.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("DIRECTORY INVENTORY")
print("=" * 140)
display(dir_df)

# ------------------------------------------------------------
# 2. JSON inventory
# ------------------------------------------------------------

json_rows = []

for p in sorted(JSON_DIR.glob("*.json")):
    obj, err = safe_read_json(p)

    if obj is None:
        json_rows.append({
            "file": p.name,
            "path": str(p),
            "status": "bad_json",
            "error": err,
        })
        continue

    json_rows.append({
        "file": p.name,
        "path": str(p),
        "stage": obj.get("stage"),
        "dataset": obj.get("dataset"),
        "group": obj.get("group"),
        "contamination": obj.get("contamination"),
        "status": obj.get("status"),
        "error": obj.get("error"),
        "n_rows": obj.get("n_rows"),
        "n_devices": obj.get("n_devices"),
        "n_features": obj.get("n_features"),
        "n_top_anomalies": obj.get("n_top_anomalies"),
        "n_top_anomaly_days": obj.get("n_top_anomaly_days"),
        "top_score": obj.get("top_score"),
        "median_score": obj.get("median_score"),
        "runtime_sec": obj.get("runtime_sec"),
        "ranked_csv": obj.get("ranked_csv"),
        "top_csv": obj.get("top_csv"),
        "device_csv": obj.get("device_csv"),
        "created_at": obj.get("created_at"),
    })

json_df = pd.DataFrame(json_rows)

json_df.to_csv(FINAL_STATS_DIR / "json_inventory.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("JSON INVENTORY")
print("=" * 140)
print("JSON files:", len(json_df))
display(json_df.head(20))

# ------------------------------------------------------------
# 3. Expected-vs-actual table
# ------------------------------------------------------------

expected_rows = []

for dataset in AGG_DATASETS:
    for group in AGG_FEATURE_GROUPS:
        for contamination in CONTAMINATION_LEVELS:
            expected_rows.append({
                "expected_stage": "agg_kkt_level",
                "prefix": "agg",
                "dataset": dataset,
                "group": group,
                "contamination": contamination,
                "expected_json": expected_json_name("agg", dataset, group, contamination),
            })

for dataset in TS_DATASETS:
    for group in TS_FEATURE_GROUPS:
        for contamination in CONTAMINATION_LEVELS:
            expected_rows.append({
                "expected_stage": "ts_kkt_day_level",
                "prefix": "ts",
                "dataset": dataset,
                "group": group,
                "contamination": contamination,
                "expected_json": expected_json_name("ts", dataset, group, contamination),
            })

expected_df = pd.DataFrame(expected_rows)

actual_map = {}
if not json_df.empty:
    for _, r in json_df.iterrows():
        actual_map[r["file"]] = r

check_rows = []

for _, r in expected_df.iterrows():
    fname = r["expected_json"]
    exists = fname in actual_map

    if exists:
        a = actual_map[fname]
        status = a.get("status")
        stage = a.get("stage")
        n_features = a.get("n_features")
        n_rows = a.get("n_rows")
        n_devices = a.get("n_devices")
        runtime_sec = a.get("runtime_sec")
        ranked_csv = a.get("ranked_csv")
        top_csv = a.get("top_csv")
        device_csv = a.get("device_csv")
        error = a.get("error")
    else:
        status = "missing"
        stage = None
        n_features = None
        n_rows = None
        n_devices = None
        runtime_sec = None
        ranked_csv = None
        top_csv = None
        device_csv = None
        error = None

    check_rows.append({
        "expected_stage": r["expected_stage"],
        "actual_stage": stage,
        "dataset": r["dataset"],
        "group": r["group"],
        "contamination": r["contamination"],
        "expected_json": fname,
        "json_exists": exists,
        "status": status,
        "n_features": n_features,
        "n_rows": n_rows,
        "n_devices": n_devices,
        "runtime_sec": runtime_sec,
        "ranked_csv_exists": Path(ranked_csv).exists() if isinstance(ranked_csv, str) else False,
        "top_csv_exists": Path(top_csv).exists() if isinstance(top_csv, str) else False,
        "device_csv_exists": Path(device_csv).exists() if isinstance(device_csv, str) else False,
        "ranked_csv": ranked_csv,
        "top_csv": top_csv,
        "device_csv": device_csv,
        "error": error,
    })

check_df = pd.DataFrame(check_rows)

check_df.to_csv(FINAL_STATS_DIR / "expected_vs_actual_experiments.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("EXPECTED VS ACTUAL")
print("=" * 140)

n_ok = int((check_df["status"] == "ok").sum())
n_missing = int((check_df["status"] == "missing").sum())
n_non_ok = int((check_df["status"] != "ok").sum())

print(f"Expected total: {EXPECTED_TOTAL}")
print(f"OK:             {n_ok}")
print(f"Missing:        {n_missing}")
print(f"Non-OK total:   {n_non_ok}")

if n_ok == EXPECTED_TOTAL:
    print("\n✅ VERDICT: ALL EXPECTED EXPERIMENTS ARE COMPLETE.")
else:
    print("\n⚠️ VERDICT: NOT ALL EXPECTED EXPERIMENTS ARE COMPLETE.")
    print("Non-OK experiments:")
    display(check_df[check_df["status"] != "ok"][[
        "expected_stage", "dataset", "group", "contamination",
        "status", "n_features", "expected_json", "error"
    ]])

print("\nBy expected stage/status:")
display(
    check_df
    .groupby(["expected_stage", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

print("\nBy dataset/status:")
display(
    check_df
    .groupby(["expected_stage", "dataset", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

# ------------------------------------------------------------
# 4. Main run summary
# ------------------------------------------------------------

ok_df = check_df[check_df["status"] == "ok"].copy()

run_summary = (
    ok_df
    .groupby(["expected_stage", "dataset", "group"], dropna=False)
    .agg(
        n_runs=("status", "count"),
        min_contamination=("contamination", "min"),
        max_contamination=("contamination", "max"),
        n_features=("n_features", "max"),
        n_rows=("n_rows", "max"),
        n_devices=("n_devices", "max"),
        total_runtime_sec=("runtime_sec", "sum"),
        mean_runtime_sec=("runtime_sec", "mean"),
        ranked_csv_all_exist=("ranked_csv_exists", "all"),
        top_csv_all_exist=("top_csv_exists", "all"),
        device_csv_any_exist=("device_csv_exists", "any"),
    )
    .reset_index()
    .sort_values(["expected_stage", "dataset", "group"])
)

run_summary.to_csv(FINAL_STATS_DIR / "run_summary_by_group.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("RUN SUMMARY BY GROUP")
print("=" * 140)
display(run_summary)

# ------------------------------------------------------------
# 5. Runtime summary
# ------------------------------------------------------------

runtime_summary = (
    ok_df
    .groupby(["expected_stage", "dataset"], dropna=False)
    .agg(
        n_runs=("status", "count"),
        total_runtime_sec=("runtime_sec", "sum"),
        mean_runtime_sec=("runtime_sec", "mean"),
        median_runtime_sec=("runtime_sec", "median"),
        max_runtime_sec=("runtime_sec", "max"),
    )
    .reset_index()
)

runtime_summary["total_runtime_min"] = runtime_summary["total_runtime_sec"] / 60
runtime_summary["total_runtime_hours"] = runtime_summary["total_runtime_sec"] / 3600

runtime_summary.to_csv(FINAL_STATS_DIR / "runtime_summary.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("RUNTIME SUMMARY")
print("=" * 140)
display(runtime_summary)

# ------------------------------------------------------------
# 6. CSV inventory
# ------------------------------------------------------------

csv_rows = []

for p in sorted(CSV_DIR.glob("*.csv")):
    head, err = safe_read_csv(p, nrows=3)
    csv_rows.append({
        "file": p.name,
        "path": str(p),
        "size_mb": round(size_mb(p), 3),
        "read_ok": err is None,
        "read_error": err,
        "sample_rows": None if head is None else len(head),
        "sample_cols": None if head is None else head.shape[1],
        "columns_preview": None if head is None else list(head.columns[:25]),
    })

csv_inventory = pd.DataFrame(csv_rows)

if not csv_inventory.empty:
    csv_inventory["kind"] = csv_inventory["file"].apply(
        lambda x:
        "agg_ranked" if x.startswith("agg__") and "__ranked" in x else
        "agg_top" if x.startswith("agg__") and "__top" in x else
        "ts_ranked_days" if x.startswith("ts__") and "__ranked_kkt_days" in x else
        "ts_top_days" if x.startswith("ts__") and "__top_kkt_days" in x else
        "ts_device_scores" if x.startswith("ts__") and "__device_aggregated_scores" in x else
        "other"
    )

csv_inventory.to_csv(FINAL_STATS_DIR / "csv_inventory.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("CSV INVENTORY")
print("=" * 140)
print("CSV files:", len(csv_inventory))
display(csv_inventory.sort_values("size_mb", ascending=False).head(50))

if not csv_inventory.empty:
    csv_kind_summary = (
        csv_inventory
        .groupby("kind")
        .agg(
            n_files=("file", "count"),
            total_mb=("size_mb", "sum"),
            max_mb=("size_mb", "max"),
            readable=("read_ok", "sum"),
        )
        .reset_index()
        .sort_values("total_mb", ascending=False)
    )
else:
    csv_kind_summary = pd.DataFrame()

csv_kind_summary.to_csv(FINAL_STATS_DIR / "csv_kind_summary.csv", index=False, encoding="utf-8-sig")

print("\nCSV kind summary:")
display(csv_kind_summary)

# ------------------------------------------------------------
# 7. Summary final files
# ------------------------------------------------------------

important_summary_files = [
    "all_kkt_anomaly_runs_summary.csv",
    "ablation_feature_group_run_summary.csv",
    "all_top_agg_kkt_anomalies_raw.csv",
    "final_agg_kkt_anomaly_ranking.csv",
    "all_top_ts_kkt_day_anomalies_raw.csv",
    "final_ts_kkt_day_anomaly_ranking.csv",
    "final_ts_device_anomaly_ranking.csv",
    "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv",
    "topk_overlap_with_all_numeric.csv",
    "KKT_ANOMALY_DETECTION_REPORT.txt",
]

summary_file_rows = []

for fname in important_summary_files:
    p = SUMMARY_DIR / fname
    row = {
        "file": fname,
        "exists": p.exists(),
        "size_mb": round(size_mb(p), 3) if p.exists() else 0.0,
        "path": str(p),
    }

    if p.exists() and p.suffix.lower() == ".csv":
        head, err = safe_read_csv(p, nrows=5)
        row["read_ok"] = err is None
        row["read_error"] = err
        row["sample_rows"] = None if head is None else len(head)
        row["sample_cols"] = None if head is None else head.shape[1]
        row["columns_preview"] = None if head is None else list(head.columns[:25])

        try:
            # Count lines cheaply.
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                row["n_lines"] = sum(1 for _ in f)
        except Exception:
            row["n_lines"] = None

    elif p.exists() and p.suffix.lower() == ".txt":
        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
            row["read_ok"] = True
            row["n_chars"] = len(text)
            row["preview"] = text[:700].replace("\n", " ")
        except Exception as e:
            row["read_ok"] = False
            row["read_error"] = repr(e)

    summary_file_rows.append(row)

summary_files_df = pd.DataFrame(summary_file_rows)
summary_files_df.to_csv(FINAL_STATS_DIR / "summary_files_inventory.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("IMPORTANT SUMMARY FILES")
print("=" * 140)
display(summary_files_df)

# ------------------------------------------------------------
# 8. Load final ranking files if available
# ------------------------------------------------------------

final_combined_path = SUMMARY_DIR / "FINAL_COMBINED_KKT_ANOMALY_RANKING.csv"
final_agg_path = SUMMARY_DIR / "final_agg_kkt_anomaly_ranking.csv"
final_ts_days_path = SUMMARY_DIR / "final_ts_kkt_day_anomaly_ranking.csv"
final_ts_device_path = SUMMARY_DIR / "final_ts_device_anomaly_ranking.csv"
overlap_path = SUMMARY_DIR / "topk_overlap_with_all_numeric.csv"

final_combined, _ = safe_read_csv(final_combined_path) if final_combined_path.exists() else (pd.DataFrame(), "missing")
final_agg, _ = safe_read_csv(final_agg_path) if final_agg_path.exists() else (pd.DataFrame(), "missing")
final_ts_days, _ = safe_read_csv(final_ts_days_path) if final_ts_days_path.exists() else (pd.DataFrame(), "missing")
final_ts_device, _ = safe_read_csv(final_ts_device_path) if final_ts_device_path.exists() else (pd.DataFrame(), "missing")
overlap_df, _ = safe_read_csv(overlap_path) if overlap_path.exists() else (pd.DataFrame(), "missing")

# ------------------------------------------------------------
# 9. Final ranking statistics
# ------------------------------------------------------------

ranking_stat_rows = []

def add_ranking_stats(name, df, score_col=None, rank_col=None, id_col="kkt_id"):
    if df is None or df.empty:
        ranking_stat_rows.append({
            "table": name,
            "exists": False,
            "n_rows": 0,
            "n_unique_kkt": 0,
            "score_col": score_col,
            "score_min": None,
            "score_median": None,
            "score_mean": None,
            "score_max": None,
        })
        return

    if score_col is None:
        candidates = [
            "final_kkt_anomaly_score",
            "final_day_anomaly_score",
            "ts_final_score",
            "anomaly_score",
            "best_score",
            "device_score_combined",
        ]
        score_col = next((c for c in candidates if c in df.columns), None)

    row = {
        "table": name,
        "exists": True,
        "n_rows": len(df),
        "n_unique_kkt": df[id_col].nunique() if id_col in df.columns else None,
        "score_col": score_col,
        "score_min": None,
        "score_median": None,
        "score_mean": None,
        "score_max": None,
    }

    if score_col is not None and score_col in df.columns:
        s = pd.to_numeric(df[score_col], errors="coerce")
        row.update({
            "score_min": float(s.min()),
            "score_median": float(s.median()),
            "score_mean": float(s.mean()),
            "score_max": float(s.max()),
            "score_p95": float(s.quantile(0.95)),
            "score_p99": float(s.quantile(0.99)),
        })

    ranking_stat_rows.append(row)

add_ranking_stats("FINAL_COMBINED_KKT_ANOMALY_RANKING", final_combined, "final_kkt_anomaly_score")
add_ranking_stats("final_agg_kkt_anomaly_ranking", final_agg, "final_kkt_anomaly_score")
add_ranking_stats("final_ts_kkt_day_anomaly_ranking", final_ts_days, "final_day_anomaly_score")
add_ranking_stats("final_ts_device_anomaly_ranking", final_ts_device, "ts_final_score")

ranking_stats_df = pd.DataFrame(ranking_stat_rows)
ranking_stats_df.to_csv(FINAL_STATS_DIR / "final_ranking_statistics.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("FINAL RANKING STATISTICS")
print("=" * 140)
display(ranking_stats_df)

# ------------------------------------------------------------
# 10. Top anomalies previews
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("TOP COMBINED KKT ANOMALIES")
print("=" * 140)

if final_combined.empty:
    print("FINAL_COMBINED_KKT_ANOMALY_RANKING.csv missing or empty.")
else:
    cols = [c for c in [
        "final_rank",
        "kkt_id",
        "final_kkt_anomaly_score",
        "best_score",
        "mean_score",
        "ts_final_score",
        "ts_total_anomaly_days",
        "ts_total_days",
        "explanation",
    ] if c in final_combined.columns]
    display(final_combined[cols].head(30))
    final_combined[cols].head(200).to_csv(FINAL_STATS_DIR / "top200_combined_kkt_anomalies.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("TOP KKT-DAY ANOMALIES")
print("=" * 140)

if final_ts_days.empty:
    print("final_ts_kkt_day_anomaly_ranking.csv missing or empty.")
else:
    cols = [c for c in [
        "final_day_rank",
        "kkt_id",
        "date",
        "final_day_anomaly_score",
        "best_day_score",
        "mean_day_score",
        "total_sum",
        "n_bills",
        "total_ecash",
        "ecash_fraction",
        "explanation",
    ] if c in final_ts_days.columns]
    display(final_ts_days[cols].head(50))
    final_ts_days[cols].head(500).to_csv(FINAL_STATS_DIR / "top500_kkt_day_anomalies.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("TOP TS DEVICE ANOMALIES")
print("=" * 140)

if final_ts_device.empty:
    print("final_ts_device_anomaly_ranking.csv missing or empty.")
else:
    cols = [c for c in [
        "kkt_id",
        "ts_final_score",
        "ts_best_device_score",
        "ts_mean_device_score",
        "ts_max_daily_score",
        "ts_total_anomaly_days",
        "ts_total_days",
        "ts_n_appearances",
    ] if c in final_ts_device.columns]
    display(final_ts_device.sort_values("ts_final_score", ascending=False)[cols].head(50))
    final_ts_device.sort_values("ts_final_score", ascending=False)[cols].head(500).to_csv(
        FINAL_STATS_DIR / "top500_ts_device_anomalies.csv",
        index=False,
        encoding="utf-8-sig"
    )

# ------------------------------------------------------------
# 11. Feature-group overlap statistics
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("TOP-K OVERLAP STATISTICS")
print("=" * 140)

if overlap_df.empty:
    print("topk_overlap_with_all_numeric.csv missing or empty.")
else:
    overlap_summary = (
        overlap_df
        .groupby(["stage", "dataset", "contamination", "group"], dropna=False)
        .agg(
            mean_jaccard=("jaccard_with_all_numeric", "mean"),
            max_jaccard=("jaccard_with_all_numeric", "max"),
            min_jaccard=("jaccard_with_all_numeric", "min"),
            mean_intersection=("intersection", "mean"),
            n_rows=("group", "count"),
        )
        .reset_index()
        .sort_values(["stage", "dataset", "contamination", "mean_jaccard"], ascending=[True, True, True, False])
    )
    overlap_summary.to_csv(FINAL_STATS_DIR / "overlap_summary.csv", index=False, encoding="utf-8-sig")
    display(overlap_summary)

# ------------------------------------------------------------
# 12. Feature-group run stats based on JSON
# ------------------------------------------------------------

print("\n" + "=" * 140)
print("FEATURE GROUP RUN STATS")
print("=" * 140)

feature_group_stats = (
    ok_df
    .groupby(["expected_stage", "dataset", "group"], dropna=False)
    .agg(
        n_contaminations=("contamination", "nunique"),
        n_features=("n_features", "max"),
        n_rows=("n_rows", "max"),
        n_devices=("n_devices", "max"),
        mean_runtime_sec=("runtime_sec", "mean"),
        total_runtime_sec=("runtime_sec", "sum"),
        all_ranked_csv_exist=("ranked_csv_exists", "all"),
        all_top_csv_exist=("top_csv_exists", "all"),
        any_device_csv_exist=("device_csv_exists", "any"),
    )
    .reset_index()
    .sort_values(["expected_stage", "dataset", "group"])
)

feature_group_stats.to_csv(FINAL_STATS_DIR / "feature_group_run_stats.csv", index=False, encoding="utf-8-sig")
display(feature_group_stats)

# ------------------------------------------------------------
# 13. Contamination-level stats
# ------------------------------------------------------------

contamination_stats = (
    ok_df
    .groupby(["expected_stage", "dataset", "contamination"], dropna=False)
    .agg(
        n_groups=("group", "nunique"),
        mean_n_features=("n_features", "mean"),
        max_n_rows=("n_rows", "max"),
        max_n_devices=("n_devices", "max"),
        total_runtime_sec=("runtime_sec", "sum"),
    )
    .reset_index()
    .sort_values(["expected_stage", "dataset", "contamination"])
)

contamination_stats.to_csv(FINAL_STATS_DIR / "contamination_level_stats.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("CONTAMINATION LEVEL STATS")
print("=" * 140)
display(contamination_stats)

# ------------------------------------------------------------
# 14. Create compact final narrative stats table
# ------------------------------------------------------------

compact_rows = []

compact_rows.append({
    "item": "Expected experiments",
    "value": EXPECTED_TOTAL,
    "note": f"{EXPECTED_AGG} aggregate + {EXPECTED_TS} time-series"
})

compact_rows.append({
    "item": "OK experiments",
    "value": n_ok,
    "note": "status == ok"
})

compact_rows.append({
    "item": "Missing/non-OK experiments",
    "value": n_non_ok,
    "note": "should be 0 for a clean final run"
})

if not ok_df.empty:
    compact_rows.append({
        "item": "Total runtime, hours",
        "value": round(ok_df["runtime_sec"].fillna(0).sum() / 3600, 3),
        "note": "sum of individual experiment runtimes"
    })

if not final_combined.empty:
    compact_rows.append({
        "item": "Combined KKT anomaly candidates",
        "value": len(final_combined),
        "note": "rows in FINAL_COMBINED_KKT_ANOMALY_RANKING.csv"
    })
    if "kkt_id" in final_combined.columns:
        compact_rows.append({
            "item": "Unique KKT in combined ranking",
            "value": final_combined["kkt_id"].nunique(),
            "note": "unique cash-register identifiers"
        })

if not final_ts_days.empty:
    compact_rows.append({
        "item": "KKT-day anomaly candidates",
        "value": len(final_ts_days),
        "note": "rows in final_ts_kkt_day_anomaly_ranking.csv"
    })
    if "kkt_id" in final_ts_days.columns:
        compact_rows.append({
            "item": "Unique KKT in day-level ranking",
            "value": final_ts_days["kkt_id"].nunique(),
            "note": "cash registers with at least one ranked anomalous day"
        })

if not final_ts_device.empty:
    compact_rows.append({
        "item": "TS device-level candidates",
        "value": len(final_ts_device),
        "note": "rows in final_ts_device_anomaly_ranking.csv"
    })

compact_final_stats = pd.DataFrame(compact_rows)
compact_final_stats.to_csv(FINAL_STATS_DIR / "compact_final_statistics.csv", index=False, encoding="utf-8-sig")

print("\n" + "=" * 140)
print("COMPACT FINAL STATISTICS")
print("=" * 140)
display(compact_final_stats)

# ------------------------------------------------------------
# 15. Write final TXT report
# ------------------------------------------------------------

txt_report = FINAL_STATS_DIR / "FINAL_STATISTICS_REPORT.txt"

with open(txt_report, "w", encoding="utf-8") as f:
    f.write("FINAL KKT ANOMALY PIPELINE STATISTICS\n")
    f.write("=" * 120 + "\n\n")
    f.write(f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"OUT_ROOT: {OUT_ROOT}\n\n")

    f.write("Completion\n")
    f.write("-" * 120 + "\n")
    f.write(f"Expected experiments: {EXPECTED_TOTAL}\n")
    f.write(f"OK experiments: {n_ok}\n")
    f.write(f"Missing / non-OK: {n_non_ok}\n")
    f.write("Verdict: " + ("ALL COMPLETE" if n_ok == EXPECTED_TOTAL else "NOT FULLY COMPLETE") + "\n\n")

    f.write("Directory inventory\n")
    f.write("-" * 120 + "\n")
    f.write(dir_df.to_string(index=False))
    f.write("\n\n")

    f.write("Run summary by group\n")
    f.write("-" * 120 + "\n")
    f.write(run_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Runtime summary\n")
    f.write("-" * 120 + "\n")
    f.write(runtime_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Final ranking statistics\n")
    f.write("-" * 120 + "\n")
    f.write(ranking_stats_df.to_string(index=False))
    f.write("\n\n")

    f.write("Compact final statistics\n")
    f.write("-" * 120 + "\n")
    f.write(compact_final_stats.to_string(index=False))
    f.write("\n\n")

    if not final_combined.empty:
        f.write("Top combined KKT anomalies\n")
        f.write("-" * 120 + "\n")
        cols = [c for c in [
            "final_rank", "kkt_id", "final_kkt_anomaly_score",
            "best_score", "mean_score", "ts_final_score", "ts_total_anomaly_days"
        ] if c in final_combined.columns]
        f.write(final_combined[cols].head(30).to_string(index=False))
        f.write("\n\n")

    if not final_ts_days.empty:
        f.write("Top KKT-day anomalies\n")
        f.write("-" * 120 + "\n")
        cols = [c for c in [
            "final_day_rank", "kkt_id", "date", "final_day_anomaly_score",
            "total_sum", "n_bills", "ecash_fraction"
        ] if c in final_ts_days.columns]
        f.write(final_ts_days[cols].head(30).to_string(index=False))
        f.write("\n\n")

    if n_non_ok > 0:
        f.write("Non-OK experiments\n")
        f.write("-" * 120 + "\n")
        f.write(check_df[check_df["status"] != "ok"][
            ["expected_stage", "dataset", "group", "contamination", "status", "n_features", "expected_json", "error"]
        ].to_string(index=False))
        f.write("\n\n")

print("\n" + "=" * 140)
print("SAVED FINAL STATISTICS")
print("=" * 140)

saved_files = [
    FINAL_STATS_DIR / "directory_inventory.csv",
    FINAL_STATS_DIR / "json_inventory.csv",
    FINAL_STATS_DIR / "expected_vs_actual_experiments.csv",
    FINAL_STATS_DIR / "run_summary_by_group.csv",
    FINAL_STATS_DIR / "runtime_summary.csv",
    FINAL_STATS_DIR / "csv_inventory.csv",
    FINAL_STATS_DIR / "csv_kind_summary.csv",
    FINAL_STATS_DIR / "summary_files_inventory.csv",
    FINAL_STATS_DIR / "final_ranking_statistics.csv",
    FINAL_STATS_DIR / "feature_group_run_stats.csv",
    FINAL_STATS_DIR / "contamination_level_stats.csv",
    FINAL_STATS_DIR / "compact_final_statistics.csv",
    FINAL_STATS_DIR / "top200_combined_kkt_anomalies.csv",
    FINAL_STATS_DIR / "top500_kkt_day_anomalies.csv",
    FINAL_STATS_DIR / "top500_ts_device_anomalies.csv",
    FINAL_STATS_DIR / "FINAL_STATISTICS_REPORT.txt",
]

for p in saved_files:
    print(("OK" if p.exists() else "MISSING"), p)

print("\n" + "=" * 140)
print("FINAL VERDICT")
print("=" * 140)

if n_ok == EXPECTED_TOTAL:
    print("✅ Всё чисто: 108/108 экспериментов OK.")
else:
    print(f"⚠️ Не всё чисто: {n_ok}/{EXPECTED_TOTAL} OK.")
    print("Смотри файл:", FINAL_STATS_DIR / "expected_vs_actual_experiments.csv")

print("Final statistics report:", txt_report)
print("\nDONE.")

FINAL KKT ANOMALY PIPELINE STATISTICS
Time: 2026-05-18 08:04:56
OUT_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs
JSON_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\results_json
CSV_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\results_csv
SUMMARY_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary
FINAL_STATS_DIR: C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS

Expected experiments:
  aggregate: 66
  time-series: 42
  total: 108

DIRECTORY INVENTORY


,folder,exists,n_files,size_mb,path
0,results_json,True,108,0.294,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
1,results_csv,True,258,45000.513,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...
2,summary,True,17,139.151,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...



JSON INVENTORY
JSON files: 108


,file,path,stage,dataset,group,contamination,status,error,n_rows,n_devices,n_features,n_top_anomalies,n_top_anomaly_days,top_score,median_score,runtime_sec,ranked_csv,top_csv,device_csv,created_at
0,agg__agg_kkt_47_11_all_numeric_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,all_numeric,0.01,ok,None,34443,NaN,150,345.0,NaN,1.0,0.159773,21.924367,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:05:18
1,agg__agg_kkt_47_11_all_numeric_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,all_numeric,0.03,ok,None,34443,NaN,150,1034.0,NaN,1.0,0.159773,18.555602,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:05:36
2,agg__agg_kkt_47_11_all_numeric_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,all_numeric,0.05,ok,None,34443,NaN,150,1723.0,NaN,1.0,0.159773,19.884701,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:05:56
3,agg__agg_kkt_47_11_benford_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,benford,0.01,ok,None,34443,NaN,1,345.0,NaN,1.0,0.141766,12.577863,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-18 07:55:04
4,agg__agg_kkt_47_11_benford_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,benford,0.03,ok,None,34443,NaN,1,1034.0,NaN,1.0,0.141766,11.839292,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-18 07:55:16
5,agg__agg_kkt_47_11_benford_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,benford,0.05,ok,None,34443,NaN,1,1723.0,NaN,1.0,0.141766,12.089234,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-18 07:55:28
6,agg__agg_kkt_47_11_bills_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,bills,0.01,ok,None,34443,NaN,11,345.0,NaN,1.0,0.158344,24.627632,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:07:50
7,agg__agg_kkt_47_11_bills_contam_0.03.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,bills,0.03,ok,None,34443,NaN,11,1034.0,NaN,1.0,0.158344,26.645549,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:08:17
8,agg__agg_kkt_47_11_bills_contam_0.05.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,bills,0.05,ok,None,34443,NaN,11,1723.0,NaN,1.0,0.158344,26.971250,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-17 16:08:44
9,agg__agg_kkt_47_11_cross_corr_contam_0.01.json,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,agg_kkt_level,agg_kkt_47_11,cross_corr,0.01,ok,None,34443,NaN,14,345.0,NaN,1.0,0.166844,30.181330,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt...,None,2026-05-18 07:55:58



EXPECTED VS ACTUAL
Expected total: 108
OK:             108
Missing:        0
Non-OK total:   0

✅ VERDICT: ALL EXPECTED EXPERIMENTS ARE COMPLETE.

By expected stage/status:


...


CONTAMINATION LEVEL STATS


,expected_stage,dataset,contamination,n_groups,mean_n_features,max_n_rows,max_n_devices,total_runtime_sec
0,agg_kkt_level,agg_kkt_47_11,0.01,11,61.363636,34443,NaN,252.383242
1,agg_kkt_level,agg_kkt_47_11,0.03,11,61.363636,34443,NaN,249.504777
2,agg_kkt_level,agg_kkt_47_11,0.05,11,61.363636,34443,NaN,247.432648
3,agg_kkt_level,agg_kkt_47_19,0.01,11,61.363636,15491,NaN,122.782898
4,agg_kkt_level,agg_kkt_47_19,0.03,11,61.363636,15491,NaN,123.773644
5,agg_kkt_level,agg_kkt_47_19,0.05,11,61.363636,15491,NaN,131.177634
6,ts_kkt_day_level,ts_kkt_47_11,0.01,7,35.714286,3168756,34443.0,2530.697720
7,ts_kkt_day_level,ts_kkt_47_11,0.03,7,35.714286,3168756,34443.0,2348.339625
8,ts_kkt_day_level,ts_kkt_47_11,0.05,7,35.714286,3168756,34443.0,2352.108239
9,ts_kkt_day_level,ts_kkt_47_19,0.01,7,35.714286,1425172,15491.0,984.402984



COMPACT FINAL STATISTICS


,item,value,note
0,Expected experiments,108.000,66 aggregate + 42 time-series
1,OK experiments,108.000,status == ok
2,Missing/non-OK experiments,0.000,should be 0 for a clean final run
3,"Total runtime, hours",3.138,sum of individual experiment runtimes
4,Combined KKT anomaly candidates,49934.000,rows in FINAL_COMBINED_KKT_ANOMALY_RANKING.csv
5,Unique KKT in combined ranking,49934.000,unique cash-register identifiers
6,KKT-day anomaly candidates,6791.000,rows in final_ts_kkt_day_anomaly_ranking.csv
7,Unique KKT in day-level ranking,1828.000,cash registers with at least one ranked anomal...
8,TS device-level candidates,49934.000,rows in final_ts_device_anomaly_ranking.csv



SAVED FINAL STATISTICS
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\directory_inventory.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\json_inventory.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\expected_vs_actual_experiments.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\run_summary_by_group.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\runtime_summary.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\csv_inventory.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\csv_kind_summary.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_outputs\summary\FINAL_STATISTICS\summary_files_inventory.csv
OK C:\Users\ivan\WORK\workshops\ICDM\Ablation\kkt_anomaly_